# Biological Age in Tohoku Medical Megabank Cohort — Biological Age Modeling with LASSO for CommCohort 67K split

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

import sys
from decimal import Decimal, ROUND_HALF_UP
import re
#For exporting .pdf file with editable text
import matplotlib
matplotlib.rcParams['pdf.fonttype']=42
matplotlib.rcParams['ps.fonttype']=42

import random
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats import multitest as multi
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

#!pip install venn
from venn import venn

#!conda install -c conda-forge -y watermark
%load_ext watermark
%watermark -a 'Akari FUJIMOTO' -udnt -vmw -iv

In [ ]:
japanize_matplotlib.japanize()

print(plt.rcParams['font.family'])

In [ ]:
#Custom function for the P-value conversion
def convert_pval(pval):
    #This function converts P-value to its annotation for a figure.
    #Of note, this function can also handle the case that P-value is zero due to the system limitation.
    #Input: P-value derived from a statistical test (float)
    #Output: P-value annotation (string)
    
    #import sys
    #from decimal import Decimal, ROUND_HALF_UP
    #import re
    
    #Check the input
    if type(pval) is str:
        text = 'Check the input object type!'
        print(text)
        return text
    if pval<0 or pval>1:
        text = 'Check whether the input value was P-value!'
        print(text)
        return text
    
    below_limit = 0#Initialize
    if pval==1.0:
        pval_text = '1.0'
    else:
        if pval==0.0:#Due to smaller than the system float minimum
            pval = sys.float_info.min
            below_limit = 1#Update
        pval_text = f'{Decimal(str(pval)):.3E}'#Take more digits because rounding is bad here
        significand, exponent = re.split('E[+\-]', pval_text)
        significand = str(Decimal(significand).quantize(Decimal('0.1'), rounding=ROUND_HALF_UP))
        if significand=='10.0':
            significand = '1.0'
            exponent = str(int(exponent)-1)
        if int(exponent)>2:
            pval_text = significand+r'$\times$'+'10'+r'$^{{-{0}}}$'.format(exponent)##Font is different in r'$ $' but \mathrm{\mathsf{{0}} doesn't work...
        elif int(exponent)>0:
            pval_text = '0.'+'0'*(int(exponent)-1)+significand.replace('.', '')
        else:
            pval_text = significand
    if below_limit==1:
        text = '<'+pval_text
    else:
        text = '='+pval_text
    
    return text

# 1. AntAge

## 1. data preparation

### 1-1. import cohort and platform

In [ ]:
#import RF
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'antDF-MF.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge-CommCohort67K-ver1_'
fileName = 'demographicsDF.tsv'

tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF

ageと身体測定値をx-score化してからその積をinteraction termとして入れる

先生の指示文だと残差になっているが、これが回帰分析なのか平均からの偏差なのか不明

#### interaction term

In [ ]:
ant_cols = ['身長 [cm]', '体重 [kg]', 'BMI [kg/m^2]']

In [ ]:

# AgeをcohortDFから取得してmetDFに一時的に追加
tempDF = metDF[ant_cols].copy()
tempDF['Age'] = cohortDF['Age']


# Age + 4身体測定項目をz-score化
scale_cols = ['Age'] + ant_cols

scaler = StandardScaler()

tempZ = pd.DataFrame(
    scaler.fit_transform(tempDF[scale_cols]),
    index=tempDF.index,
    columns=scale_cols)

cols = []
# interaction termを作成
for col in ant_cols:
    tempZ[f'{col}*年齢[year]'] = tempZ[col] * tempZ['Age']
    cols.append(f'{col}*年齢[year]')

tempZ = tempZ.drop(columns='Age')
scaler = StandardScaler()
tempZ[cols] = scaler.fit_transform(tempZ[cols])

display(tempZ)
display(tempZ.describe(include='all'))

# metDFを上書き
metDF = tempZ


### 1-2. stratification with sex

In [ ]:
#Stratify the study cohort with sex
cohortDF_F = cohortDF.loc[cohortDF['Sex']=='Female']
cohortDF_M = cohortDF.loc[cohortDF['Sex']=='Male']
cohortDF_P = cohortDF#Not copy just rename
cohortD = {'SexPooled':cohortDF_P, 'Female':cohortDF_F, 'Male':cohortDF_M}

for cohort in cohortD.keys():
    tempDF = cohortD[cohort]
    print(cohort+': ', len(tempDF))

In [ ]:
tempDF1 = metDF
tempD1 = cohortD

#Prepare data per sex-stratified cohort
tempD2 = {}
for cohort in tempD1.keys():
    tempI = tempD1[cohort].index
    tempDF = tempDF1.loc[tempDF1.index.isin(tempI)]
    print(cohort+': ', len(tempDF))
    tempD2[cohort] = tempDF

metD = tempD2

### 1-3 split cohort

In [ ]:
tempD1 = cohortD
nmodels = 10#Fix throughout this notebook
random.seed(123)#For reproducibility (while checking the following independency)

tempD2 = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]
    
    #Randomize the row order
    tempL = tempDF1.index.tolist()
    tempL = random.sample(tempL, len(tempL))
    tempDF = tempDF1.loc[tempL]
    
    #Split cohort to define the training and testing (hold-out) sets
    tempL1 = np.array_split(tempDF, nmodels)#List of DFs
    ##Save only the testing set assignment
    tempL2 = []
    for model_k in range(nmodels):
        tempDF = tempL1[model_k]
        model_n = 'Model'+str(model_k+1).zfill(2)
        tempS = pd.Series(np.repeat(model_n, len(tempDF)), index=tempDF.index, name='Testing')
        tempL2.append(tempS)
    tempS = pd.concat(tempL2, axis=0)
    
    #Add to the original metadata
    tempDF = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    tempD2[cohort] = tempDF
    print(cohort)
    display(tempDF)
    display(tempDF['Testing'].value_counts())
    print('')

cohortD = tempD2#Update

### 1-4 check independency about sets

In [ ]:
tempD1 = cohortD
tempL0 = ['Testing', 'Sex', 'Age', 'BMI']#Set only the variables used in the study

for cohort in tempD1.keys():
    tempDF = tempD1[cohort]
    tempDF = tempDF[tempL0]
    
    tempL1 = []#For variable name
    tempL2 = []#Test name
    tempL3 = []#For degrees of freedom
    tempL4 = []#For test statistic
    tempL5 = []#For P-value
    
    #Categorical variables
    tempL = tempDF.select_dtypes(exclude=[np.number]).columns.tolist()
    for col_n in tempL:
        if col_n!='Testing':
            if (col_n=='Sex')&(cohort in ['Female', 'Male']):
                continue
            else:
                tempDF1 = pd.crosstab(tempDF[col_n], tempDF['Testing'])
                #Pearson's chi-squared test
                chi2, pval, dof, tempA = stats.chi2_contingency(tempDF1, correction=False)
                tempL1.append(col_n)
                tempL2.append('Pearson\'s chi-squared test')
                tempL3.append(dof)
                tempL4.append(chi2)
                tempL5.append(pval)
    
    #Numeric variables
    tempL = tempDF.select_dtypes(include=[np.number]).columns.tolist()
    for col_n in tempL:
        #ANOVA
        formula = col_n+' ~ C(Testing)'
        model = smf.ols(formula, data=tempDF).fit()
        tempDF1 = anova_lm(model, typ=1)#ANOVA type doesn't matter this case
        tempL1.append(col_n)
        tempL2.append('ANOVA')
        dof1 = tempDF1['df'].astype('int64').loc['C(Testing)']#Between-groups
        dof2 = tempDF1['df'].astype('int64').loc['Residual']#Within-groups
        tempL3.append((dof1, dof2))
        tempL4.append(tempDF1['F'].loc['C(Testing)'])
        tempL5.append(tempDF1['PR(>F)'].loc['C(Testing)'])

    #Summarize results as a single DF
    tempDF = pd.DataFrame({'Variable':tempL1,
                           'StatisticalTest':tempL2,
                           'N':len(tempDF),
                           'DoF':tempL3,
                           'Statistic':tempL4,
                           'Pval':tempL5})
    tempDF = tempDF.set_index('Variable')
    #P-value adjustment by using Benjamini–Hochberg method
    tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                            is_sorted=False, returnsorted=False)[1]
    print(cohort)
    display(tempDF)
    
    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_SpeAge-LASSO_'
    fileName = 'ANT_split-sets-independency-'+cohort+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

## 2. generate Bage model

### 2-1 lasso modeling with CV

In [ ]:
tempD1 = cohortD
tempD2 = metD
yvar = 'Age'#Fix throughout this notebook
ralphas = 0.05#Length of the regularization path (= alpha_min / alpha_max)
nalphas = 200#Number of alphas along the regularization path
ncvs = 10#Fix throughout this notebook
ncpus = -1#All processors

#Modeling per sex-stratified cohort
summaryD = {}#Initialize
bcoefD = {}#Initialize
predictD = {}#Initialize
agegapD = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]#Dependent variable and info about testing set
    tempDF2 = tempD2[cohort]#Independent variables
    t_start1 = time.time()
    print(cohort+' start:', time.ctime(t_start1))
    
    #Modeling per model number (testing set)
    tempL1 = []#For alpha
    #PAge
    tempL2 = []#For the coefficient of determination R2
    tempL3 = []#For beta-coefficients
    tempL4 = []#For predictions
    #other
    tempL7 = []#Agegap_reg
    tempL8 = []#BAge
    tempL9 = []#BAge R2score
    tempL10 = []#regression
    
    for model_k in range(nmodels):
        t_start2 = time.time()
        #Prepare DFs for model k
        model_n = 'Model'+str(model_k+1).zfill(2)
        ##Training set
        tempI = tempDF1.loc[tempDF1['Testing']!=model_n].index
        yDF_train = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_train = tempDF2.loc[tempI]
        ##Testing (hold-out) set
        tempI = tempDF1.loc[tempDF1['Testing']==model_n].index
        yDF_test = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_test = tempDF2.loc[tempI]
        
        #Fit model with cross-validation using training dataset (i.e., internal training/validation datasets)
        model = LassoCV(eps=ralphas, n_alphas=nalphas, alphas=None, fit_intercept=True,
                        precompute='auto', cv=ncvs, n_jobs=ncpus)
        model.fit(xDF_train, yDF_train)
        
        #Save alpha
        tempL1.append(model.alpha_)#Penalization amount decided by cross-validation
        
        #Evaluation with the testing (hold-out) set
        tempL2.append(model.score(xDF_test, yDF_test))
        
        #Save beta-coefficients
        tempS = pd.Series(model.coef_,#w in the cost function formula
                          index=tempDF2.columns, name=model_n)
        tempS.loc['Intercept'] = model.intercept_#Independent term in decision function
        tempS.index.name = 'Variable'
        tempL3.append(tempS)

        #PAge prediction, agegap, BAge
        #train PAge prediction
        train_P = pd.Series(model.predict(xDF_train),index=xDF_train.index,name='PAge')
        #regression PAge~age -> calculate agegap
        train_g = pd.DataFrame({'Age': yDF_train['Age'],
                                'PAge':train_P})
        reg_fit = smf.ols('PAge ~ Age', data=train_g).fit()
        tempL10.append(pd.Series({'Intercept':reg_fit.params['Intercept'],
                                'Age': reg_fit.params['Age']}, name=model_n))

        #test PAge prediction
        test_P = pd.Series(model.predict(xDF_test),index=xDF_test.index,name='PAge')
        #test calculate agegap
        test_g = pd.DataFrame({'Age': yDF_test['Age'],
                             'PAge': test_P})
        test_g['AgeGap'] = (test_g['PAge'] - reg_fit.predict(test_g))
        #test: BAge = Age + AgeGap
        test_g['BAge'] = (test_g['Age'] + test_g['AgeGap'])
        
        #save PAge prediction
        tempL4.append(test_P)
        #save AgeGap and BAge
        tempL7.append(test_g['AgeGap'].rename('AgeGap'))
        tempL8.append(test_g['BAge'].rename('BAge'))

        #calculate BAge R2
        r2_b = 1- ((test_g['Age'] - test_g['BAge'])**2).sum()/(
            (test_g['Age'] - test_g['Age'].mean())**2).sum()
        tempL9.append(r2_b)
        
        t_elapsed2 = time.time() - t_start2
        print('- '+model_n+':',
              round(t_elapsed2//60), 'min', round(t_elapsed2%60, 1), 'sec')
    t_elapsed1 = time.time() - t_start1
    print('- Total elapsed time for '+str(nmodels)+' models with '+str(ncvs)+'-fold CV:',
          round(t_elapsed1//60), 'min', round(t_elapsed1%60, 1), 'sec')
    
    #Summarize results across models into a single DF
    tempL = ['Model'+str(model_k+1).zfill(2) for model_k in range(nmodels)]
    summaryDF = pd.DataFrame({'L1alpha':tempL1, 'R2score_P':tempL2, 'R2score_B':tempL9},
                             index=pd.Index(tempL, name='ModelID'))
    display(summaryDF)
    bcoefDF = pd.concat(tempL3, axis=1)
    display(bcoefDF)
    tempS1 = tempDF1[yvar]
    tempS2 = pd.concat(tempL4, axis=0)#PAge
    tempS3 = pd.concat(tempL7, axis=0)#agegap
    tempS4 = pd.concat(tempL8, axis=0)#BAge
    predictDF = pd.concat([tempS1, tempS2, tempS3, tempS4], axis=1)
    display(predictDF)
    agegapDF = pd.DataFrame(tempL10)
    agegapDF.index.name = 'ModelID'
    agegapDF = agegapDF.T
    agegapDF.columns.name = 'ModelID'
    display(agegapDF)
    print('')

        #Save (Of note, bcoefDF is to be saved later)
    fileDir = './ExportData//Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_AntAge-LASSO_'
    fileName = 'model-summary-'+cohort+'.tsv'
    summaryDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'bcoefs-'+cohort+'.tsv'
    #bcoefDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    fileName = 'predictions-'+cohort+'.tsv'
    predictDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_mean-'+cohort+'.tsv'
    #SmeanDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_sd-'+cohort+'.tsv'
    #SsdDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'gap_regression-'+cohort+'.tsv'
    #agegapDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    

    summaryD[cohort] = summaryDF
    bcoefD[cohort] = bcoefDF
    predictD[cohort] = predictDF
    agegapD[cohort] = agegapDF

### 2-2. prediction accuracy

In [ ]:
#Summary
tempD1 = summaryD
tempD2 = predictD

met = {'R2score_P':'PAge', 'R2score_B':'BAge'}

for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        print(model+' model')
        tempDF1 = tempD1[model]
        tempS = tempDF1[metric]
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SD]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1))#Sample standard deviation
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SEM]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1)/np.sqrt(len(tempS)))
        tempDF2 = tempD2[model]
        print(' - Observed vs. predicted '+yvar+': (Pearson\'s r, P) =',
              stats.pearsonr(tempDF2[yvar], tempDF2[yvar_model]))
        tempDF1 = tempDF1.describe()
        tempDF2 = tempDF2.describe()
        tempDF = pd.merge(tempDF1, tempDF2, left_index=True, right_index=True, how='left')
        display(tempDF)

In [ ]:
tempD1 = summaryD
met = {'R2score_P':'PAge', 'R2score_B':'BAge'}
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}

#Prepare DF
tempD = {}
for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        tempDF = tempD1[model]
        tempD[model] = tempDF[metric].tolist()
    tempDF = pd.DataFrame(tempD)
    display(tempDF.describe())
    tempDF = tempDF.melt(var_name='Model', value_name='R2', value_vars=tempDF.columns)

    #Plot
    sns.set(style='ticks',  context='talk')
    plt.figure(figsize=(2, 3))
    sns.barplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
            hue='Model', hue_order=tempD2.keys(), dodge=False, palette=tempD2,
            ci=95, capsize=0.4, errwidth=1.5, errcolor='black', edgecolor='black')
    p = sns.stripplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
                  hue='Model', hue_order=tempD2.keys(), dodge=False, jitter=0.3,
                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    sns.despine()
    plt.setp(p.get_xticklabels(), rotation=70,
         horizontalalignment='right', verticalalignment='center', rotation_mode='anchor')
    plt.ylabel('Out-of-sample '+r'$R^2$_'+yvar_model, fontsize=13)
    plt.xlabel('')
    plt.show()

In [ ]:
tempD1 = predictD

for metric, yvar_model in met.items():
#Prepare plot range
    range_min = np.min([df[var].min() for df in tempD1.values() for var in [yvar, yvar_model]])
    range_max = np.max([df[var].max() for df in tempD1.values() for var in [yvar, yvar_model]])
    offset_rate = 0.05
    offset = (range_max-range_min)*offset_rate
    range_min = range_min - offset#Update
    range_max = range_max + offset#Update
    xymin = 20
    xymax = 100
    xyinter = 20

#Plot measured vs. predicted per model
    tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = 'Age [years]'
    sns.set(style='ticks', context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(10, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        model = list(tempD2.keys())[ax_i]
        tempDF = tempD1[model]
    #Regplot
        sns.regplot(data=tempDF, y=yvar_model, x=yvar, color=tempD2[model],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Predicted '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF[yvar], tempDF[yvar_model])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(model, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Measured '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

#Plot difference b/w sex-specific and sex-pooled models
    tempD2 = {'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = yvar_model+' [years]'
    sns.set(style='ticks',  context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(6.5, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        sex = list(tempD2.keys())[ax_i]
    #Prepare DF
        tempS1 = tempD1[sex][yvar_model]
        tempS1.name = 'Sex-specific'
        tempS2 = tempD1['SexPooled'][yvar_model]
        tempS2.name = 'Sex-pooled'
        tempDF = pd.merge(tempS1, tempS2, left_index=True, right_index=True, how='inner')
    #Regplot
        sns.regplot(data=tempDF, x='Sex-specific', y='Sex-pooled', color=tempD2[sex],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Sex-pooled '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF['Sex-specific'], tempDF['Sex-pooled'])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(sex, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Sex-specific '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

## 3. model predictors

### 3-1. clean beta-coeficiesnt dataframaea

In [ ]:
tempD1 = bcoefD.copy()

tempD2 = {}
for model in tempD1.keys():
    tempDF = tempD1[model]

    #Calculate standard statistic
    tempS1 = tempDF.mean(axis=1)
    tempS1.name = 'Mean'
    tempS2 = tempDF.std(axis=1, ddof=1)#Sample standard deviation
    tempS2.name = 'SD'
    tempS3 = (tempDF==0.0).sum(axis=1)
    tempS3.name = 'nZeros'
    tempDF = pd.concat([tempDF, tempS1, tempS2, tempS3], axis=1)

    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_AntAge-LASSO_'
    fileName = 'bcoefs-'+model+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    tempD2[model] = tempDF
    
    #Check
    tempDF2 = tempDF.loc[['Intercept']]#Retrieve as pd.DataFrame
    tempDF = tempDF.drop(index=['Intercept'])
    print(model+' model:')
    print(' - Variables:', len(tempDF))
    tempDF1 = tempDF.loc[tempDF['nZeros']!=10]
    print(' - Variables with non-zero beta-coefficient:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF.loc[tempDF['nZeros']==0]
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF1.sort_values(by='Mean', ascending=False)
    display(tempDF1)
    display(tempDF2)#Intercept
    print('')

bcoefD = tempD2#Update

### 3-2 valiables retained in all models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Variables with non-zero beta-coefficient in all models
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:')
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF))
    
    #Prepare DF for plot
    tempDF = tempDF.sort_values(by='Mean', ascending=False)
    tempL = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    
    #Plot the robustly retained variables
    sns.set(style='ticks', context='talk')
    japanize_matplotlib.japanize()
    plt.figure(figsize=(7, len(tempL)/3))
    sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                dodge=False, color=tempD2[model], saturation=1,
                showcaps=True, notch=False, showfliers=False)
    sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                  markers='x', color='black',
                  dodge=False, join=False, estimator=np.mean, ci=None)#Add only the mean
    p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL, dodge=False, jitter=0.3,
                      size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    #p.grid(axis='x', linestyle='--', color='black')
    p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
    sns.despine()
    p.set(xlim=(-7.5, 8), xticks=np.arange(-7, 8.1, 1))#Fixed across models
    p.set(ylim=(-0.5, len(tempL)-0.5), xticks=np.arange(-7, 8.1, 1))
    ##Change default dull line color of sns.boxplot (saturation parameter is for patch)
    for line in p.get_lines():
        line.set_color('black')
    for box in p.artists:
        box.set_edgecolor('black')
    ##Add background color
    for row_i in range(len(tempL)):
        if row_i%2==0:
            plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
    p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
          title=tempD3[model]+' model (robust predictors)')
    plt.show()
    print('')

### 3-3. Comparison between sex-pooled and sex-specific models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare the robust predictors
tempD = {}
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempD[model] = set(tempDF.index.tolist())

#Venn diagram
sns.set(style='ticks',  context='talk')
fig, ax = plt.subplots(figsize=(4, 4))
venn(tempD, fmt='{size:,}', cmap=list(tempD2.values()), legend_loc=None, ax=ax)
plt.setp(ax, ylim=(0.025, 0.925))#Otherwise, weird space...
##Add legend annotation
x_coord = [0.3, 0.7, 0.65]
y_coord = [0.75, 0.75, 0.25]
h_align = ['right', 'left', 'left']
v_align = ['bottom', 'bottom', 'top']
for i in range(len(tempD3)):
    key = list(tempD3.keys())[i]
    total = f'{len(tempD[key]):,}'
    ax.text(x_coord[i], y_coord[i], tempD3[key]+'\n('+total+' items)',
            fontsize='small', multialignment='center',
            horizontalalignment=h_align[i], verticalalignment=v_align[i],
            bbox={'boxstyle':'round', 'facecolor':tempD2[key], 'pad':0.2, 'alpha':0.5})
title = 'Robustly retained items'
ax.set_title(title, fontsize='medium')
plt.show()

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare union of the robust predictors
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempL.append(tempDF.index.to_series())
tempI = pd.concat(tempL, axis=0)
tempL1 = tempI.unique().tolist()
print('-> Union:', len(tempL1))

#Prepare DF for plot
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Take the target variables
    tempDF = tempDF.loc[tempL1]
    
    #Prepare DF for plot
    if model=='SexPooled':
        tempDF = tempDF.sort_values(by='Mean', ascending=False)
        tempL2 = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    tempDF['Model'] = model
    tempL.append(tempDF)
tempDF = pd.concat(tempL, axis=0)

#Plot the target variables
sns.set(style='ticks', context='talk')
japanize_matplotlib.japanize()
plt.figure(figsize=(10, len(tempL2)))
p = sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
            hue='Model', hue_order=tempD2.keys(), dodge=True, palette=tempD2, saturation=1,
            showcaps=True, notch=False, showfliers=False)
#sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#              hue='Model', hue_order=tempD2.keys(), dodge=True, markers='x', color='black',
#              join=False, estimator=np.mean, ci=None)#Add only the mean
#p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#                  hue='Model', hue_order=tempD2.keys(), dodge=True, jitter=0.3,
#                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
#p.grid(axis='x', linestyle='--', color='black')
p.legend(bbox_to_anchor=(1.02, 1),loc='upper left')
p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
sns.despine()
p.set(xlim=(-8.5, 8), xticks=np.arange(-8, 8.1, 1))#Fixed across models
p.set(ylim=(-0.5, len(tempL2)-0.5), xticks=np.arange(-8, 8.1, 1))
##Change default dull line color of sns.boxplot (saturation parameter is for patch)
for line in p.get_lines():
    line.set_color('black')
for box in p.artists:
    box.set_edgecolor('black')
##Add background color
for row_i in range(len(tempL2)):
    if row_i%2==0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
      title=tempD3[model]+' model (robust predictors)')
plt.show()

### 3-4. Explained variance in CAge by the variable retained in ≥1 model

OLS linear regression model for significance: Age \~ b0 + b1\*Variable + b2\*Sex + b3\*BMI  
Univariate model: Age ~ b0 + b1\*Variable  

Of note, biological age is included for comparison. As well as LASSO models, unstandardized dependent variable is used to set the same beta-coefficient scale.  

In [ ]:
japanize_matplotlib.japanize()
tempD1 = bcoefD
tempD2 = metD
tempD3 = predictD
tempD4 = cohortD
covarL = ['Sex', 'BMI']#Covariates (for now)
formula1 = yvar+' ~ Variable'#Univariate model
formula2 = yvar+' ~ Variable + C(Sex) + BMI'#Full model
formula3 = yvar+' ~ Variable + BMI'#Full model for the case that sex is invariant
tempD5 = {'SexPooled':formula2, 'Female':formula3, 'Male':formula3}
topX = 30

for metric, yvar_model in met.items():
    print('============================='+yvar_model+'=============================')
    for model in tempD1.keys():
        print(model+' model:')
    
        #Variables with non-zero beta-coefficient at least one of the models
        tempDF = tempD1[model]
        tempL = tempDF.loc[tempDF['nZeros']==10].index.tolist()#To be excluded
    
    #Prepare DF for regressions
        tempDF1 = tempD2[model].drop(columns=tempL)
    ##Add model predictions for comparison
        tempDF = tempD3[model]
        tempDF1[yvar_model+' model'] = tempDF[yvar_model]
        tempL = tempDF1.columns.tolist()#Save independent variables (including model) for regression-loop
    ##Add covariates
        tempDF = tempD4[model]
        tempDF = tempDF[covarL]
        tempDF1 = pd.merge(tempDF1, tempDF, left_index=True, right_index=True, how='left')
    ##Z-score transformation
        tempDF2 = tempDF1.select_dtypes(include=[np.number])
        scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
        tempA = scaler.fit_transform(tempDF2)#Column direction
        tempDF2 = pd.DataFrame(data=tempA, index=tempDF2.index, columns=tempDF2.columns)
    ###Recover categorical covariates
        tempDF3 = tempDF1.select_dtypes(exclude=[np.number])
        tempDF1 = pd.merge(tempDF2, tempDF3, left_index=True, right_index=True, how='left')
        ##Add unstandardized dependent variable
        tempDF = tempD4[model]
        tempS = tempDF[yvar]
        tempDF1 = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    #Perform OLS linear regression
        tempL1 = []#For sample size
        tempL2 = []#For R2 in the univariate model
        tempL3 = []#For R2
        tempL4 = []#For beta-coefficient
        tempL5 = []#For SE of beta-coefficient
        tempL6 = []#For t-statistic
        tempL7 = []#For residual degrees of freedom
        tempL8 = []#For P-value
        t_start = time.time()
        for var in tempL:
        #Rename independent variable
            tempDF = tempDF1.rename(columns={var: 'Variable'})
        #Add a constant for the intercept -> Similar to R, smf automatically add a constant
        
        #Save sample size
            tempL1.append(len(tempDF))
        
        #Fit univariate model
            fit_res = smf.ols(formula1, data=tempDF).fit()
        #Save R2 [%]
            tempL2.append(fit_res.rsquared*100)

        #Fit full model
            formula = tempD5[model]
            fit_res = smf.ols(formula, data=tempDF).fit()
        #Save R2 [%]
            tempL3.append(fit_res.rsquared*100)
        #Save beta-coefficient of the variable
            tempL4.append(fit_res.params['Variable'])
            tempL5.append(fit_res.bse['Variable'])
        #Save t-statistic of the variable
            tempL6.append(fit_res.tvalues['Variable'])
        #Save residual degrees of freedom
            tempL7.append(int(fit_res.df_resid))
        #Save P-value of the variable
            tempL8.append(fit_res.pvalues['Variable'])
        t_elapsed = time.time() - t_start
        print('Elapsed time for', len(tempL), 'OLS linear regressions (including '+yvar_model+'):',
              round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
        
    #Clean the results
        tempDF1 = pd.DataFrame({'N':tempL1, 'UnivarR2':tempL2, 'R2':tempL3,
                            'OLSbcoef':tempL4, 'OLSbcoefSE':tempL5,
                            'tStat':tempL6, 'DoF':tempL7,  'Pval':tempL8},
                           index=pd.Index(tempL, name='Variable'))
    ##P-value adjustment by using Benjamini–Hochberg method
        tempDF1['AdjPval'] = multi.multipletests(tempDF1['Pval'], alpha=0.05, method='fdr_bh',
                                             is_sorted=False, returnsorted=False)[1]
    ##Add the LASSO results
        tempDF = tempD1[model]
        tempDF1['LASSObcoef'] = tempDF.loc[tempDF.index.isin(tempL)]['Mean']#NaN in model reference
        tempDF1['LASSOnZeros'] = tempDF.loc[tempDF.index.isin(tempL)]['nZeros']#NaN in model reference
        tempDF1 = tempDF1.sort_values(by='AdjPval', ascending=True)
    
    #Save the cleaned DF
    #fileDir = './ExportData/67k/'
    #ipynbName = '260907_TMM-BAge_MetAge-LASSO-CommCohort67K-ver2k_'
    #fileName = 'OLS-'+model+'.tsv'
    #tempDF1.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    #Extact significant variables
        tempDF = tempDF1[tempDF1['AdjPval']<0.05]
        print('Variables significantly associated with '+yvar+' (FDR < 0.05):', len(tempDF)-1)#model reference
    ##Top X (+ model reference)
        tempDF = tempDF.iloc[0:(topX+1)]
        display(tempDF)
        ##Category and color
        tempL = []
        for row_n in tempDF.index:
            if row_n==yvar_model+' model':
                tempL.append('Positive association')
            else:
                bcoef = tempDF.loc[row_n, 'OLSbcoef']
                if bcoef>0:
                    tempL.append('Positive association')
                elif bcoef<0:
                    tempL.append('Negative association')
                else:
                    tempL.append('Error?')
        tempDF['Association'] = tempL
        tempD = {'Positive association':'tab:red', 'Negative association':'tab:blue'}
         ##Plot
        tempDF = tempDF.reset_index()
        sns.set(style='ticks',  context='talk')
        plt.figure(figsize=(10, 4))
        p = sns.barplot(data=tempDF, y='UnivarR2', x='Variable', hue='Association', hue_order=tempD.keys(),
                    dodge=False, palette=tempD, edgecolor='k')
        p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))#Fixed across models
        p.grid(axis='y', linestyle='--', color='black')
        sns.despine()
        plt.xticks(rotation=90, horizontalalignment='center')
        plt.xlabel('')
        plt.ylabel('Explained variance in '+yvar+' [%]',fontsize=15)
        plt.legend(loc='upper right')
        plt.show()
        print('')

# 2. HemAge

## 1. data preparation

### 1-1. import cohort and platform

In [ ]:
#import RF met HC
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'hemDF-MF.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge-CommCohort67K-ver1_'
fileName = 'demographicsDF.tsv'

tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF

### 1-2. stratification with sex

In [ ]:
#Stratify the study cohort with sex
cohortDF_F = cohortDF.loc[cohortDF['Sex']=='Female']
cohortDF_M = cohortDF.loc[cohortDF['Sex']=='Male']
cohortDF_P = cohortDF#Not copy just rename
cohortD = {'SexPooled':cohortDF_P, 'Female':cohortDF_F, 'Male':cohortDF_M}

for cohort in cohortD.keys():
    tempDF = cohortD[cohort]
    print(cohort+': ', len(tempDF))

In [ ]:
tempDF1 = metDF
tempD1 = cohortD

#Prepare data per sex-stratified cohort
tempD2 = {}
for cohort in tempD1.keys():
    tempI = tempD1[cohort].index
    tempDF = tempDF1.loc[tempDF1.index.isin(tempI)]
    print(cohort+': ', len(tempDF))
    tempD2[cohort] = tempDF

metD = tempD2

### 1-3 split cohort

In [ ]:
tempD1 = cohortD
nmodels = 10#Fix throughout this notebook
random.seed(123)#For reproducibility (while checking the following independency)

tempD2 = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]
    
    #Randomize the row order
    tempL = tempDF1.index.tolist()
    tempL = random.sample(tempL, len(tempL))
    tempDF = tempDF1.loc[tempL]
    
    #Split cohort to define the training and testing (hold-out) sets
    tempL1 = np.array_split(tempDF, nmodels)#List of DFs
    ##Save only the testing set assignment
    tempL2 = []
    for model_k in range(nmodels):
        tempDF = tempL1[model_k]
        model_n = 'Model'+str(model_k+1).zfill(2)
        tempS = pd.Series(np.repeat(model_n, len(tempDF)), index=tempDF.index, name='Testing')
        tempL2.append(tempS)
    tempS = pd.concat(tempL2, axis=0)
    
    #Add to the original metadata
    tempDF = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    tempD2[cohort] = tempDF
    print(cohort)
    display(tempDF)
    display(tempDF['Testing'].value_counts())
    print('')

cohortD = tempD2#Update

### 1-4 check independency about sets

In [ ]:
tempD1 = cohortD
tempL0 = ['Testing', 'Sex', 'Age', 'BMI']#Set only the variables used in the study

for cohort in tempD1.keys():
    tempDF = tempD1[cohort]
    tempDF = tempDF[tempL0]
    
    tempL1 = []#For variable name
    tempL2 = []#Test name
    tempL3 = []#For degrees of freedom
    tempL4 = []#For test statistic
    tempL5 = []#For P-value
    
    #Categorical variables
    tempL = tempDF.select_dtypes(exclude=[np.number]).columns.tolist()
    for col_n in tempL:
        if col_n!='Testing':
            if (col_n=='Sex')&(cohort in ['Female', 'Male']):
                continue
            else:
                tempDF1 = pd.crosstab(tempDF[col_n], tempDF['Testing'])
                #Pearson's chi-squared test
                chi2, pval, dof, tempA = stats.chi2_contingency(tempDF1, correction=False)
                tempL1.append(col_n)
                tempL2.append('Pearson\'s chi-squared test')
                tempL3.append(dof)
                tempL4.append(chi2)
                tempL5.append(pval)
    
    #Numeric variables
    tempL = tempDF.select_dtypes(include=[np.number]).columns.tolist()
    for col_n in tempL:
        #ANOVA
        formula = col_n+' ~ C(Testing)'
        model = smf.ols(formula, data=tempDF).fit()
        tempDF1 = anova_lm(model, typ=1)#ANOVA type doesn't matter this case
        tempL1.append(col_n)
        tempL2.append('ANOVA')
        dof1 = tempDF1['df'].astype('int64').loc['C(Testing)']#Between-groups
        dof2 = tempDF1['df'].astype('int64').loc['Residual']#Within-groups
        tempL3.append((dof1, dof2))
        tempL4.append(tempDF1['F'].loc['C(Testing)'])
        tempL5.append(tempDF1['PR(>F)'].loc['C(Testing)'])

    #Summarize results as a single DF
    tempDF = pd.DataFrame({'Variable':tempL1,
                           'StatisticalTest':tempL2,
                           'N':len(tempDF),
                           'DoF':tempL3,
                           'Statistic':tempL4,
                           'Pval':tempL5})
    tempDF = tempDF.set_index('Variable')
    #P-value adjustment by using Benjamini–Hochberg method
    tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                            is_sorted=False, returnsorted=False)[1]
    print(cohort)
    display(tempDF)
    
    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261005_TMM-BAge_SpeAge-LASSO_'
    fileName = 'HEM_split-sets-independency-'+cohort+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

## 2. generate Bage model

### 2-1 lasso modeling with CV

In [ ]:
tempD1 = cohortD
tempD2 = metD
yvar = 'Age'#Fix throughout this notebook
ralphas = 0.05#Length of the regularization path (= alpha_min / alpha_max)
nalphas = 200#Number of alphas along the regularization path
ncvs = 10#Fix throughout this notebook
ncpus = -1#All processors

#Modeling per sex-stratified cohort
summaryD = {}#Initialize
bcoefD = {}#Initialize
predictD = {}#Initialize
scaler_meanD = {} #Inirialize
scaler_sdD={}
agegapD = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]#Dependent variable and info about testing set
    tempDF2 = tempD2[cohort]#Independent variables
    t_start1 = time.time()
    print(cohort+' start:', time.ctime(t_start1))
    
    #Modeling per model number (testing set)
    tempL1 = []#For alpha
    #PAge
    tempL2 = []#For the coefficient of determination R2
    tempL3 = []#For beta-coefficients
    tempL4 = []#For predictions
    tempL5=[] #For scaler means
    tempL6=[]#For scaler scales
    #other
    tempL7 = []#Agegap_reg
    tempL8 = []#BAge
    tempL9 = []#BAge R2score
    tempL10 = []#regression
    
    for model_k in range(nmodels):
        t_start2 = time.time()
        #Prepare DFs for model k
        model_n = 'Model'+str(model_k+1).zfill(2)
        ##Training set
        tempI = tempDF1.loc[tempDF1['Testing']!=model_n].index
        yDF_train = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_train = tempDF2.loc[tempI]
        ##Testing (hold-out) set
        tempI = tempDF1.loc[tempDF1['Testing']==model_n].index
        yDF_test = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_test = tempDF2.loc[tempI]
        
        #Standardization (only for independent variables)
        ##Fit with the training set
        scaler_x = StandardScaler(copy=True, with_mean=True, with_std=True).fit(xDF_train)#Column direction
        
        #Save scaler parameters
        tempL5.append(pd.Series(scaler_x.mean_, index=xDF_train.columns,name=model_n))
        tempL6.append(pd.Series(scaler_x.scale_,index=xDF_train.columns,name=model_n))
        
        ##Z-score transformation using the training set-fitted scaler
        tempA = scaler_x.transform(xDF_train)
        xDF_train = pd.DataFrame(data=tempA, index=xDF_train.index, columns=xDF_train.columns)#Update
        tempA = scaler_x.transform(xDF_test)
        xDF_test = pd.DataFrame(data=tempA, index=xDF_test.index, columns=xDF_test.columns)#Update
        
        #Fit model with cross-validation using training dataset (i.e., internal training/validation datasets)
        model = LassoCV(eps=ralphas, n_alphas=nalphas, alphas=None, fit_intercept=True,
                        precompute='auto', cv=ncvs, n_jobs=ncpus)
        model.fit(xDF_train, yDF_train)
        
        #Save alpha
        tempL1.append(model.alpha_)#Penalization amount decided by cross-validation
        
        #Evaluation with the testing (hold-out) set
        tempL2.append(model.score(xDF_test, yDF_test))
        
        #Save beta-coefficients
        tempS = pd.Series(model.coef_,#w in the cost function formula
                          index=tempDF2.columns, name=model_n)
        tempS.loc['Intercept'] = model.intercept_#Independent term in decision function
        tempS.index.name = 'Variable'
        tempL3.append(tempS)

        #PAge prediction, agegap, BAge
        #train PAge prediction
        train_P = pd.Series(model.predict(xDF_train),index=xDF_train.index,name='PAge')
        #regression PAge~age -> calculate agegap
        train_g = pd.DataFrame({'Age': yDF_train['Age'],
                                'PAge':train_P})
        reg_fit = smf.ols('PAge ~ Age', data=train_g).fit()
        tempL10.append(pd.Series({'Intercept':reg_fit.params['Intercept'],
                                'Age': reg_fit.params['Age']}, name=model_n))

        #test PAge prediction
        test_P = pd.Series(model.predict(xDF_test),index=xDF_test.index,name='PAge')
        #test calculate agegap
        test_g = pd.DataFrame({'Age': yDF_test['Age'],
                             'PAge': test_P})
        test_g['AgeGap'] = (test_g['PAge'] - reg_fit.predict(test_g))
        #test: BAge = Age + AgeGap
        test_g['BAge'] = (test_g['Age'] + test_g['AgeGap'])
        
        #save PAge prediction
        tempL4.append(test_P)
        #save AgeGap and BAge
        tempL7.append(test_g['AgeGap'].rename('AgeGap'))
        tempL8.append(test_g['BAge'].rename('BAge'))

        #calculate BAge R2
        r2_b = 1- ((test_g['Age'] - test_g['BAge'])**2).sum()/(
            (test_g['Age'] - test_g['Age'].mean())**2).sum()
        tempL9.append(r2_b)
        
        t_elapsed2 = time.time() - t_start2
        print('- '+model_n+':',
              round(t_elapsed2//60), 'min', round(t_elapsed2%60, 1), 'sec')
    t_elapsed1 = time.time() - t_start1
    print('- Total elapsed time for '+str(nmodels)+' models with '+str(ncvs)+'-fold CV:',
          round(t_elapsed1//60), 'min', round(t_elapsed1%60, 1), 'sec')
    
    #Summarize results across models into a single DF
    tempL = ['Model'+str(model_k+1).zfill(2) for model_k in range(nmodels)]
    summaryDF = pd.DataFrame({'L1alpha':tempL1, 'R2score_P':tempL2, 'R2score_B':tempL9},
                             index=pd.Index(tempL, name='ModelID'))
    display(summaryDF)
    bcoefDF = pd.concat(tempL3, axis=1)
    display(bcoefDF)
    tempS1 = tempDF1[yvar]
    tempS2 = pd.concat(tempL4, axis=0)#PAge
    tempS3 = pd.concat(tempL7, axis=0)#agegap
    tempS4 = pd.concat(tempL8, axis=0)#BAge
    predictDF = pd.concat([tempS1, tempS2, tempS3, tempS4], axis=1)
    display(predictDF)
    SmeanDF= pd.concat(tempL5, axis=1)
    SmeanDF.columns.name = 'ModelID'
    display(SmeanDF)
    SsdDF = pd.concat(tempL6, axis=1)
    SsdDF.columns.name = 'ModelID'
    display(SsdDF)
    agegapDF = pd.DataFrame(tempL10)
    agegapDF.index.name = 'ModelID'
    agegapDF = agegapDF.T
    agegapDF.columns.name = 'ModelID'
    display(agegapDF)
    print('')
    
    #Save (Of note, bcoefDF is to be saved later)
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_HemAge-LASSO_'
    fileName = 'model-summary-'+cohort+'.tsv'
    summaryDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'bcoefs-'+cohort+'.tsv'
    #bcoefDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    fileName = 'predictions-'+cohort+'.tsv'
    predictDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_mean-'+cohort+'.tsv'
    #SmeanDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_sd-'+cohort+'.tsv'
    #SsdDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'gap_regression-'+cohort+'.tsv'
    #agegapDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    
    summaryD[cohort] = summaryDF
    bcoefD[cohort] = bcoefDF
    predictD[cohort] = predictDF
    scaler_meanD[cohort]= SmeanDF
    scaler_sdD[cohort] = SsdDF
    agegapD[cohort] = agegapDF

### 2-2. prediction accuracy

In [ ]:
#Summary
tempD1 = summaryD
tempD2 = predictD

met = {'R2score_P':'PAge', 'R2score_B':'BAge'}

for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        print(model+' model')
        tempDF1 = tempD1[model]
        tempS = tempDF1[metric]
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SD]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1))#Sample standard deviation
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SEM]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1)/np.sqrt(len(tempS)))
        tempDF2 = tempD2[model]
        print(' - Observed vs. predicted '+yvar+': (Pearson\'s r, P) =',
              stats.pearsonr(tempDF2[yvar], tempDF2[yvar_model]))
        tempDF1 = tempDF1.describe()
        tempDF2 = tempDF2.describe()
        tempDF = pd.merge(tempDF1, tempDF2, left_index=True, right_index=True, how='left')
        display(tempDF)

In [ ]:
tempD1 = summaryD
met = {'R2score_P':'PAge', 'R2score_B':'BAge'}
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}

#Prepare DF
tempD = {}
for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        tempDF = tempD1[model]
        tempD[model] = tempDF[metric].tolist()
    tempDF = pd.DataFrame(tempD)
    display(tempDF.describe())
    tempDF = tempDF.melt(var_name='Model', value_name='R2', value_vars=tempDF.columns)

    #Plot
    sns.set(style='ticks',  context='talk')
    plt.figure(figsize=(2, 3))
    sns.barplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
            hue='Model', hue_order=tempD2.keys(), dodge=False, palette=tempD2,
            ci=95, capsize=0.4, errwidth=1.5, errcolor='black', edgecolor='black')
    p = sns.stripplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
                  hue='Model', hue_order=tempD2.keys(), dodge=False, jitter=0.3,
                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    sns.despine()
    plt.setp(p.get_xticklabels(), rotation=70,
         horizontalalignment='right', verticalalignment='center', rotation_mode='anchor')
    plt.ylabel('Out-of-sample '+r'$R^2$_'+yvar_model, fontsize=13)
    plt.xlabel('')
    plt.show()

In [ ]:
tempD1 = predictD

for metric, yvar_model in met.items():
#Prepare plot range
    range_min = np.min([df[var].min() for df in tempD1.values() for var in [yvar, yvar_model]])
    range_max = np.max([df[var].max() for df in tempD1.values() for var in [yvar, yvar_model]])
    offset_rate = 0.05
    offset = (range_max-range_min)*offset_rate
    range_min = range_min - offset#Update
    range_max = range_max + offset#Update
    xymin = 20
    xymax = 100
    xyinter = 20

#Plot measured vs. predicted per model
    tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = 'Age [years]'
    sns.set(style='ticks', context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(10, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        model = list(tempD2.keys())[ax_i]
        tempDF = tempD1[model]
    #Regplot
        sns.regplot(data=tempDF, y=yvar_model, x=yvar, color=tempD2[model],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Predicted '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF[yvar], tempDF[yvar_model])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(model, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Measured '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

#Plot difference b/w sex-specific and sex-pooled models
    tempD2 = {'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = yvar_model+' [years]'
    sns.set(style='ticks',  context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(6.5, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        sex = list(tempD2.keys())[ax_i]
    #Prepare DF
        tempS1 = tempD1[sex][yvar_model]
        tempS1.name = 'Sex-specific'
        tempS2 = tempD1['SexPooled'][yvar_model]
        tempS2.name = 'Sex-pooled'
        tempDF = pd.merge(tempS1, tempS2, left_index=True, right_index=True, how='inner')
    #Regplot
        sns.regplot(data=tempDF, x='Sex-specific', y='Sex-pooled', color=tempD2[sex],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Sex-pooled '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF['Sex-specific'], tempDF['Sex-pooled'])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(sex, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Sex-specific '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

## 3. model predictors

### 3-1. clean beta-coeficiesnt dataframaea

In [ ]:
tempD1 = bcoefD.copy()

tempD2 = {}
for model in tempD1.keys():
    tempDF = tempD1[model]

    #Calculate standard statistic
    tempS1 = tempDF.mean(axis=1)
    tempS1.name = 'Mean'
    tempS2 = tempDF.std(axis=1, ddof=1)#Sample standard deviation
    tempS2.name = 'SD'
    tempS3 = (tempDF==0.0).sum(axis=1)
    tempS3.name = 'nZeros'
    tempDF = pd.concat([tempDF, tempS1, tempS2, tempS3], axis=1)

    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_HemAge-LASSO_'
    fileName = 'bcoefs-'+model+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    tempD2[model] = tempDF
    
    #Check
    tempDF2 = tempDF.loc[['Intercept']]#Retrieve as pd.DataFrame
    tempDF = tempDF.drop(index=['Intercept'])
    print(model+' model:')
    print(' - Variables:', len(tempDF))
    tempDF1 = tempDF.loc[tempDF['nZeros']!=10]
    print(' - Variables with non-zero beta-coefficient:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF.loc[tempDF['nZeros']==0]
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF1.sort_values(by='Mean', ascending=False)
    display(tempDF1)
    display(tempDF2)#Intercept
    print('')

bcoefD = tempD2#Update

### 3-2 valiables retained in all models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Variables with non-zero beta-coefficient in all models
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:')
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF))
    
    #Prepare DF for plot
    tempDF = tempDF.sort_values(by='Mean', ascending=False)
    tempL = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    
    #Plot the robustly retained variables
    sns.set(style='ticks', context='talk')
    japanize_matplotlib.japanize()
    plt.figure(figsize=(7, len(tempL)/3))
    sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                dodge=False, color=tempD2[model], saturation=1,
                showcaps=True, notch=False, showfliers=False)
    sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                  markers='x', color='black',
                  dodge=False, join=False, estimator=np.mean, ci=None)#Add only the mean
    p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL, dodge=False, jitter=0.3,
                      size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    #p.grid(axis='x', linestyle='--', color='black')
    p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
    sns.despine()
    p.set(xlim=(-5.5, 2), xticks=np.arange(-5, 2.1, 1))#Fixed across models
    p.set(ylim=(-0.5, len(tempL)-0.5), xticks=np.arange(-5, 2.1, 1))
    ##Change default dull line color of sns.boxplot (saturation parameter is for patch)
    for line in p.get_lines():
        line.set_color('black')
    for box in p.artists:
        box.set_edgecolor('black')
    ##Add background color
    for row_i in range(len(tempL)):
        if row_i%2==0:
            plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
    p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
          title=tempD3[model]+' model (robust predictors)')
    plt.show()
    print('')

### 3-3. Comparison between sex-pooled and sex-specific models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare the robust predictors
tempD = {}
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempD[model] = set(tempDF.index.tolist())

#Venn diagram
sns.set(style='ticks',  context='talk')
fig, ax = plt.subplots(figsize=(4, 4))
venn(tempD, fmt='{size:,}', cmap=list(tempD2.values()), legend_loc=None, ax=ax)
plt.setp(ax, ylim=(0.025, 0.925))#Otherwise, weird space...
##Add legend annotation
x_coord = [0.3, 0.7, 0.65]
y_coord = [0.75, 0.75, 0.25]
h_align = ['right', 'left', 'left']
v_align = ['bottom', 'bottom', 'top']
for i in range(len(tempD3)):
    key = list(tempD3.keys())[i]
    total = f'{len(tempD[key]):,}'
    ax.text(x_coord[i], y_coord[i], tempD3[key]+'\n('+total+' items)',
            fontsize='small', multialignment='center',
            horizontalalignment=h_align[i], verticalalignment=v_align[i],
            bbox={'boxstyle':'round', 'facecolor':tempD2[key], 'pad':0.2, 'alpha':0.5})
title = 'Robustly retained items'
ax.set_title(title, fontsize='medium')
plt.show()

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare union of the robust predictors
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempL.append(tempDF.index.to_series())
tempI = pd.concat(tempL, axis=0)
tempL1 = tempI.unique().tolist()
print('-> Union:', len(tempL1))

#Prepare DF for plot
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Take the target variables
    tempDF = tempDF.loc[tempL1]
    
    #Prepare DF for plot
    if model=='SexPooled':
        tempDF = tempDF.sort_values(by='Mean', ascending=False)
        tempL2 = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    tempDF['Model'] = model
    tempL.append(tempDF)
tempDF = pd.concat(tempL, axis=0)

#Plot the target variables
sns.set(style='ticks', context='talk')
japanize_matplotlib.japanize()
plt.figure(figsize=(8, len(tempL2)/3))
p = sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
            hue='Model', hue_order=tempD2.keys(), dodge=True, palette=tempD2, saturation=1,
            showcaps=True, notch=False, showfliers=False)
#sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#              hue='Model', hue_order=tempD2.keys(), dodge=True, markers='x', color='black',
#              join=False, estimator=np.mean, ci=None)#Add only the mean
#p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#                  hue='Model', hue_order=tempD2.keys(), dodge=True, jitter=0.3,
#                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
#p.grid(axis='x', linestyle='--', color='black')
p.legend(bbox_to_anchor=(1.02, 1),loc='upper left')
p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
sns.despine()
p.set(xlim=(-4.5, 3), xticks=np.arange(-4, 3.1, 1))#Fixed across models
p.set(ylim=(-0.5, len(tempL2)-0.5), xticks=np.arange(-4, 3.1, 1))
##Change default dull line color of sns.boxplot (saturation parameter is for patch)
for line in p.get_lines():
    line.set_color('black')
for box in p.artists:
    box.set_edgecolor('black')
##Add background color
for row_i in range(len(tempL2)):
    if row_i%2==0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
      title=tempD3[model]+' model (robust predictors)')
plt.show()

### 3-4. Explained variance in CAge by the variable retained in ≥1 model

OLS linear regression model for significance: Age \~ b0 + b1\*Variable + b2\*Sex + b3\*BMI  
Univariate model: Age ~ b0 + b1\*Variable  

Of note, biological age is included for comparison. As well as LASSO models, unstandardized dependent variable is used to set the same beta-coefficient scale.  

In [ ]:

tempD1 = bcoefD
tempD2 = metD
tempD3 = predictD
tempD4 = cohortD
covarL = ['Sex', 'BMI']#Covariates (for now)
formula1 = yvar+' ~ Variable'#Univariate model
formula2 = yvar+' ~ Variable + C(Sex) + BMI'#Full model
formula3 = yvar+' ~ Variable + BMI'#Full model for the case that sex is invariant
tempD5 = {'SexPooled':formula2, 'Female':formula3, 'Male':formula3}
topX = 30

for metric, yvar_model in met.items():
    print('============================='+yvar_model+'=============================')
    for model in tempD1.keys():
        print(model+' model:')
    
        #Variables with non-zero beta-coefficient at least one of the models
        tempDF = tempD1[model]
        tempL = tempDF.loc[tempDF['nZeros']==10].index.tolist()#To be excluded
    
    #Prepare DF for regressions
        tempDF1 = tempD2[model].drop(columns=tempL)
    ##Add model predictions for comparison
        tempDF = tempD3[model]
        tempDF1[yvar_model+' model'] = tempDF[yvar_model]
        tempL = tempDF1.columns.tolist()#Save independent variables (including model) for regression-loop
    ##Add covariates
        tempDF = tempD4[model]
        tempDF = tempDF[covarL]
        tempDF1 = pd.merge(tempDF1, tempDF, left_index=True, right_index=True, how='left')
    ##Z-score transformation
        tempDF2 = tempDF1.select_dtypes(include=[np.number])
        scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
        tempA = scaler.fit_transform(tempDF2)#Column direction
        tempDF2 = pd.DataFrame(data=tempA, index=tempDF2.index, columns=tempDF2.columns)
    ###Recover categorical covariates
        tempDF3 = tempDF1.select_dtypes(exclude=[np.number])
        tempDF1 = pd.merge(tempDF2, tempDF3, left_index=True, right_index=True, how='left')
        ##Add unstandardized dependent variable
        tempDF = tempD4[model]
        tempS = tempDF[yvar]
        tempDF1 = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    #Perform OLS linear regression
        tempL1 = []#For sample size
        tempL2 = []#For R2 in the univariate model
        tempL3 = []#For R2
        tempL4 = []#For beta-coefficient
        tempL5 = []#For SE of beta-coefficient
        tempL6 = []#For t-statistic
        tempL7 = []#For residual degrees of freedom
        tempL8 = []#For P-value
        t_start = time.time()
        for var in tempL:
        #Rename independent variable
            tempDF = tempDF1.rename(columns={var: 'Variable'})
        #Add a constant for the intercept -> Similar to R, smf automatically add a constant
        
        #Save sample size
            tempL1.append(len(tempDF))
        
        #Fit univariate model
            fit_res = smf.ols(formula1, data=tempDF).fit()
        #Save R2 [%]
            tempL2.append(fit_res.rsquared*100)

        #Fit full model
            formula = tempD5[model]
            fit_res = smf.ols(formula, data=tempDF).fit()
        #Save R2 [%]
            tempL3.append(fit_res.rsquared*100)
        #Save beta-coefficient of the variable
            tempL4.append(fit_res.params['Variable'])
            tempL5.append(fit_res.bse['Variable'])
        #Save t-statistic of the variable
            tempL6.append(fit_res.tvalues['Variable'])
        #Save residual degrees of freedom
            tempL7.append(int(fit_res.df_resid))
        #Save P-value of the variable
            tempL8.append(fit_res.pvalues['Variable'])
        t_elapsed = time.time() - t_start
        print('Elapsed time for', len(tempL), 'OLS linear regressions (including '+yvar_model+'):',
              round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
        
    #Clean the results
        tempDF1 = pd.DataFrame({'N':tempL1, 'UnivarR2':tempL2, 'R2':tempL3,
                            'OLSbcoef':tempL4, 'OLSbcoefSE':tempL5,
                            'tStat':tempL6, 'DoF':tempL7,  'Pval':tempL8},
                           index=pd.Index(tempL, name='Variable'))
    ##P-value adjustment by using Benjamini–Hochberg method
        tempDF1['AdjPval'] = multi.multipletests(tempDF1['Pval'], alpha=0.05, method='fdr_bh',
                                             is_sorted=False, returnsorted=False)[1]
    ##Add the LASSO results
        tempDF = tempD1[model]
        tempDF1['LASSObcoef'] = tempDF.loc[tempDF.index.isin(tempL)]['Mean']#NaN in model reference
        tempDF1['LASSOnZeros'] = tempDF.loc[tempDF.index.isin(tempL)]['nZeros']#NaN in model reference
        tempDF1 = tempDF1.sort_values(by='AdjPval', ascending=True)
    
    #Save the cleaned DF
    #fileDir = './ExportData/67k/'
    #ipynbName = '260907_TMM-BAge_MetAge-LASSO-CommCohort67K-ver2k_'
    #fileName = 'OLS-'+model+'.tsv'
    #tempDF1.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    #Extact significant variables
        tempDF = tempDF1[tempDF1['AdjPval']<0.05]
        print('Variables significantly associated with '+yvar+' (FDR < 0.05):', len(tempDF)-1)#model reference
    ##Top X (+ model reference)
        tempDF = tempDF.iloc[0:(topX+1)]
        display(tempDF)
        ##Category and color
        tempL = []
        for row_n in tempDF.index:
            if row_n==yvar_model+' model':
                tempL.append('Positive association')
            else:
                bcoef = tempDF.loc[row_n, 'OLSbcoef']
                if bcoef>0:
                    tempL.append('Positive association')
                elif bcoef<0:
                    tempL.append('Negative association')
                else:
                    tempL.append('Error?')
        tempDF['Association'] = tempL
        tempD = {'Positive association':'tab:red', 'Negative association':'tab:blue'}
         ##Plot
        tempDF = tempDF.reset_index()
        sns.set(style='ticks',  context='talk')
        plt.figure(figsize=(10, 4))
        p = sns.barplot(data=tempDF, y='UnivarR2', x='Variable', hue='Association', hue_order=tempD.keys(),
                    dodge=False, palette=tempD, edgecolor='k')
        p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))#Fixed across models
        p.grid(axis='y', linestyle='--', color='black')
        sns.despine()
        plt.xticks(rotation=90, horizontalalignment='center')
        plt.xlabel('')
        plt.ylabel('Explained variance in '+yvar+' [%]',fontsize=15)
        plt.legend(loc='upper right')
        japanize_matplotlib.japanize()
        plt.show()
        print('')

# 3. ChemAge

## 1. data preparation

### 1-1. import cohort and platform

In [ ]:
#import RF met HC
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'chemDF-MF.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge-CommCohort67K-ver1_'
fileName = 'demographicsDF.tsv'

tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF

### 1-2. stratification with sex

In [ ]:
#Stratify the study cohort with sex
cohortDF_F = cohortDF.loc[cohortDF['Sex']=='Female']
cohortDF_M = cohortDF.loc[cohortDF['Sex']=='Male']
cohortDF_P = cohortDF#Not copy just rename
cohortD = {'SexPooled':cohortDF_P, 'Female':cohortDF_F, 'Male':cohortDF_M}

for cohort in cohortD.keys():
    tempDF = cohortD[cohort]
    print(cohort+': ', len(tempDF))

In [ ]:
tempDF1 = metDF
tempD1 = cohortD

#Prepare data per sex-stratified cohort
tempD2 = {}
for cohort in tempD1.keys():
    tempI = tempD1[cohort].index
    tempDF = tempDF1.loc[tempDF1.index.isin(tempI)]
    print(cohort+': ', len(tempDF))
    tempD2[cohort] = tempDF

metD = tempD2

### 1-3 split cohort

In [ ]:
tempD1 = cohortD
nmodels = 10#Fix throughout this notebook
random.seed(123)#For reproducibility (while checking the following independency)

tempD2 = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]
    
    #Randomize the row order
    tempL = tempDF1.index.tolist()
    tempL = random.sample(tempL, len(tempL))
    tempDF = tempDF1.loc[tempL]
    
    #Split cohort to define the training and testing (hold-out) sets
    tempL1 = np.array_split(tempDF, nmodels)#List of DFs
    ##Save only the testing set assignment
    tempL2 = []
    for model_k in range(nmodels):
        tempDF = tempL1[model_k]
        model_n = 'Model'+str(model_k+1).zfill(2)
        tempS = pd.Series(np.repeat(model_n, len(tempDF)), index=tempDF.index, name='Testing')
        tempL2.append(tempS)
    tempS = pd.concat(tempL2, axis=0)
    
    #Add to the original metadata
    tempDF = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    tempD2[cohort] = tempDF
    print(cohort)
    display(tempDF)
    display(tempDF['Testing'].value_counts())
    print('')

cohortD = tempD2#Update

### 1-4 check independency about sets

In [ ]:
tempD1 = cohortD
tempL0 = ['Testing', 'Sex', 'Age', 'BMI']#Set only the variables used in the study

for cohort in tempD1.keys():
    tempDF = tempD1[cohort]
    tempDF = tempDF[tempL0]
    
    tempL1 = []#For variable name
    tempL2 = []#Test name
    tempL3 = []#For degrees of freedom
    tempL4 = []#For test statistic
    tempL5 = []#For P-value
    
    #Categorical variables
    tempL = tempDF.select_dtypes(exclude=[np.number]).columns.tolist()
    for col_n in tempL:
        if col_n!='Testing':
            if (col_n=='Sex')&(cohort in ['Female', 'Male']):
                continue
            else:
                tempDF1 = pd.crosstab(tempDF[col_n], tempDF['Testing'])
                #Pearson's chi-squared test
                chi2, pval, dof, tempA = stats.chi2_contingency(tempDF1, correction=False)
                tempL1.append(col_n)
                tempL2.append('Pearson\'s chi-squared test')
                tempL3.append(dof)
                tempL4.append(chi2)
                tempL5.append(pval)
    
    #Numeric variables
    tempL = tempDF.select_dtypes(include=[np.number]).columns.tolist()
    for col_n in tempL:
        #ANOVA
        formula = col_n+' ~ C(Testing)'
        model = smf.ols(formula, data=tempDF).fit()
        tempDF1 = anova_lm(model, typ=1)#ANOVA type doesn't matter this case
        tempL1.append(col_n)
        tempL2.append('ANOVA')
        dof1 = tempDF1['df'].astype('int64').loc['C(Testing)']#Between-groups
        dof2 = tempDF1['df'].astype('int64').loc['Residual']#Within-groups
        tempL3.append((dof1, dof2))
        tempL4.append(tempDF1['F'].loc['C(Testing)'])
        tempL5.append(tempDF1['PR(>F)'].loc['C(Testing)'])

    #Summarize results as a single DF
    tempDF = pd.DataFrame({'Variable':tempL1,
                           'StatisticalTest':tempL2,
                           'N':len(tempDF),
                           'DoF':tempL3,
                           'Statistic':tempL4,
                           'Pval':tempL5})
    tempDF = tempDF.set_index('Variable')
    #P-value adjustment by using Benjamini–Hochberg method
    tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                            is_sorted=False, returnsorted=False)[1]
    print(cohort)
    display(tempDF)
    
    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261005_TMM-BAge_SpeAge-LASSO_'
    fileName = 'CHEM_split-sets-independency-'+cohort+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

## 2. generate Bage model

### 2-1 lasso modeling with CV

In [ ]:
tempD1 = cohortD
tempD2 = metD
yvar = 'Age'#Fix throughout this notebook
ralphas = 0.05#Length of the regularization path (= alpha_min / alpha_max)
nalphas = 200#Number of alphas along the regularization path
ncvs = 10#Fix throughout this notebook
ncpus = -1#All processors

#Modeling per sex-stratified cohort
summaryD = {}#Initialize
bcoefD = {}#Initialize
predictD = {}#Initialize
scaler_meanD = {} #Inirialize
scaler_sdD={}
agegapD = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]#Dependent variable and info about testing set
    tempDF2 = tempD2[cohort]#Independent variables
    t_start1 = time.time()
    print(cohort+' start:', time.ctime(t_start1))
    
    #Modeling per model number (testing set)
    tempL1 = []#For alpha
    #PAge
    tempL2 = []#For the coefficient of determination R2
    tempL3 = []#For beta-coefficients
    tempL4 = []#For predictions
    tempL5=[] #For scaler means
    tempL6=[]#For scaler scales
    #other
    tempL7 = []#Agegap_reg
    tempL8 = []#BAge
    tempL9 = []#BAge R2score
    tempL10 = []#regression
    
    for model_k in range(nmodels):
        t_start2 = time.time()
        #Prepare DFs for model k
        model_n = 'Model'+str(model_k+1).zfill(2)
        ##Training set
        tempI = tempDF1.loc[tempDF1['Testing']!=model_n].index
        yDF_train = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_train = tempDF2.loc[tempI]
        ##Testing (hold-out) set
        tempI = tempDF1.loc[tempDF1['Testing']==model_n].index
        yDF_test = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_test = tempDF2.loc[tempI]
        
        #Standardization (only for independent variables)
        ##Fit with the training set
        scaler_x = StandardScaler(copy=True, with_mean=True, with_std=True).fit(xDF_train)#Column direction
        
        #Save scaler parameters
        tempL5.append(pd.Series(scaler_x.mean_, index=xDF_train.columns,name=model_n))
        tempL6.append(pd.Series(scaler_x.scale_,index=xDF_train.columns,name=model_n))
        
        ##Z-score transformation using the training set-fitted scaler
        tempA = scaler_x.transform(xDF_train)
        xDF_train = pd.DataFrame(data=tempA, index=xDF_train.index, columns=xDF_train.columns)#Update
        tempA = scaler_x.transform(xDF_test)
        xDF_test = pd.DataFrame(data=tempA, index=xDF_test.index, columns=xDF_test.columns)#Update
        
        #Fit model with cross-validation using training dataset (i.e., internal training/validation datasets)
        model = LassoCV(eps=ralphas, n_alphas=nalphas, alphas=None, fit_intercept=True,
                        precompute='auto', cv=ncvs, n_jobs=ncpus)
        model.fit(xDF_train, yDF_train)
        
        #Save alpha
        tempL1.append(model.alpha_)#Penalization amount decided by cross-validation
        
        #Evaluation with the testing (hold-out) set
        tempL2.append(model.score(xDF_test, yDF_test))
        
        #Save beta-coefficients
        tempS = pd.Series(model.coef_,#w in the cost function formula
                          index=tempDF2.columns, name=model_n)
        tempS.loc['Intercept'] = model.intercept_#Independent term in decision function
        tempS.index.name = 'Variable'
        tempL3.append(tempS)

        #PAge prediction, agegap, BAge
        #train PAge prediction
        train_P = pd.Series(model.predict(xDF_train),index=xDF_train.index,name='PAge')
        #regression PAge~age -> calculate agegap
        train_g = pd.DataFrame({'Age': yDF_train['Age'],
                                'PAge':train_P})
        reg_fit = smf.ols('PAge ~ Age', data=train_g).fit()
        tempL10.append(pd.Series({'Intercept':reg_fit.params['Intercept'],
                                'Age': reg_fit.params['Age']}, name=model_n))

        #test PAge prediction
        test_P = pd.Series(model.predict(xDF_test),index=xDF_test.index,name='PAge')
        #test calculate agegap
        test_g = pd.DataFrame({'Age': yDF_test['Age'],
                             'PAge': test_P})
        test_g['AgeGap'] = (test_g['PAge'] - reg_fit.predict(test_g))
        #test: BAge = Age + AgeGap
        test_g['BAge'] = (test_g['Age'] + test_g['AgeGap'])
        
        #save PAge prediction
        tempL4.append(test_P)
        #save AgeGap and BAge
        tempL7.append(test_g['AgeGap'].rename('AgeGap'))
        tempL8.append(test_g['BAge'].rename('BAge'))

        #calculate BAge R2
        r2_b = 1- ((test_g['Age'] - test_g['BAge'])**2).sum()/(
            (test_g['Age'] - test_g['Age'].mean())**2).sum()
        tempL9.append(r2_b)
        
        t_elapsed2 = time.time() - t_start2
        print('- '+model_n+':',
              round(t_elapsed2//60), 'min', round(t_elapsed2%60, 1), 'sec')
    t_elapsed1 = time.time() - t_start1
    print('- Total elapsed time for '+str(nmodels)+' models with '+str(ncvs)+'-fold CV:',
          round(t_elapsed1//60), 'min', round(t_elapsed1%60, 1), 'sec')
    
    #Summarize results across models into a single DF
    tempL = ['Model'+str(model_k+1).zfill(2) for model_k in range(nmodels)]
    summaryDF = pd.DataFrame({'L1alpha':tempL1, 'R2score_P':tempL2, 'R2score_B':tempL9},
                             index=pd.Index(tempL, name='ModelID'))
    display(summaryDF)
    bcoefDF = pd.concat(tempL3, axis=1)
    display(bcoefDF)
    tempS1 = tempDF1[yvar]
    tempS2 = pd.concat(tempL4, axis=0)#PAge
    tempS3 = pd.concat(tempL7, axis=0)#agegap
    tempS4 = pd.concat(tempL8, axis=0)#BAge
    predictDF = pd.concat([tempS1, tempS2, tempS3, tempS4], axis=1)
    display(predictDF)
    SmeanDF= pd.concat(tempL5, axis=1)
    SmeanDF.columns.name = 'ModelID'
    display(SmeanDF)
    SsdDF = pd.concat(tempL6, axis=1)
    SsdDF.columns.name = 'ModelID'
    display(SsdDF)
    agegapDF = pd.DataFrame(tempL10)
    agegapDF.index.name = 'ModelID'
    agegapDF = agegapDF.T
    agegapDF.columns.name = 'ModelID'
    display(agegapDF)
    print('')
    
    #Save (Of note, bcoefDF is to be saved later)
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_ChemAge-LASSO_'
    fileName = 'model-summary-'+cohort+'.tsv'
    summaryDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'bcoefs-'+cohort+'.tsv'
    #bcoefDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    fileName = 'predictions-'+cohort+'.tsv'
    predictDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_mean-'+cohort+'.tsv'
    #SmeanDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_sd-'+cohort+'.tsv'
    #SsdDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'gap_regression-'+cohort+'.tsv'
    #agegapDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    
    summaryD[cohort] = summaryDF
    bcoefD[cohort] = bcoefDF
    predictD[cohort] = predictDF
    scaler_meanD[cohort]= SmeanDF
    scaler_sdD[cohort] = SsdDF
    agegapD[cohort] = agegapDF

### 2-2. prediction accuracy

In [ ]:
#Summary
tempD1 = summaryD
tempD2 = predictD

met = {'R2score_P':'PAge', 'R2score_B':'BAge'}

for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        print(model+' model')
        tempDF1 = tempD1[model]
        tempS = tempDF1[metric]
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SD]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1))#Sample standard deviation
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SEM]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1)/np.sqrt(len(tempS)))
        tempDF2 = tempD2[model]
        print(' - Observed vs. predicted '+yvar+': (Pearson\'s r, P) =',
              stats.pearsonr(tempDF2[yvar], tempDF2[yvar_model]))
        tempDF1 = tempDF1.describe()
        tempDF2 = tempDF2.describe()
        tempDF = pd.merge(tempDF1, tempDF2, left_index=True, right_index=True, how='left')
        display(tempDF)

In [ ]:
tempD1 = summaryD
met = {'R2score_P':'PAge', 'R2score_B':'BAge'}
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}

#Prepare DF
tempD = {}
for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        tempDF = tempD1[model]
        tempD[model] = tempDF[metric].tolist()
    tempDF = pd.DataFrame(tempD)
    display(tempDF.describe())
    tempDF = tempDF.melt(var_name='Model', value_name='R2', value_vars=tempDF.columns)

    #Plot
    sns.set(style='ticks',  context='talk')
    plt.figure(figsize=(2, 3))
    sns.barplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
            hue='Model', hue_order=tempD2.keys(), dodge=False, palette=tempD2,
            ci=95, capsize=0.4, errwidth=1.5, errcolor='black', edgecolor='black')
    p = sns.stripplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
                  hue='Model', hue_order=tempD2.keys(), dodge=False, jitter=0.3,
                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    sns.despine()
    plt.setp(p.get_xticklabels(), rotation=70,
         horizontalalignment='right', verticalalignment='center', rotation_mode='anchor')
    plt.ylabel('Out-of-sample '+r'$R^2$_'+yvar_model, fontsize=13)
    plt.xlabel('')
    plt.show()

In [ ]:
tempD1 = predictD

for metric, yvar_model in met.items():
#Prepare plot range
    range_min = np.min([df[var].min() for df in tempD1.values() for var in [yvar, yvar_model]])
    range_max = np.max([df[var].max() for df in tempD1.values() for var in [yvar, yvar_model]])
    offset_rate = 0.05
    offset = (range_max-range_min)*offset_rate
    range_min = range_min - offset#Update
    range_max = range_max + offset#Update
    xymin = 20
    xymax = 100
    xyinter = 20

#Plot measured vs. predicted per model
    tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = 'Age [years]'
    sns.set(style='ticks', context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(10, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        model = list(tempD2.keys())[ax_i]
        tempDF = tempD1[model]
    #Regplot
        sns.regplot(data=tempDF, y=yvar_model, x=yvar, color=tempD2[model],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Predicted '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF[yvar], tempDF[yvar_model])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(model, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Measured '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

#Plot difference b/w sex-specific and sex-pooled models
    tempD2 = {'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = yvar_model+' [years]'
    sns.set(style='ticks',  context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(6.5, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        sex = list(tempD2.keys())[ax_i]
    #Prepare DF
        tempS1 = tempD1[sex][yvar_model]
        tempS1.name = 'Sex-specific'
        tempS2 = tempD1['SexPooled'][yvar_model]
        tempS2.name = 'Sex-pooled'
        tempDF = pd.merge(tempS1, tempS2, left_index=True, right_index=True, how='inner')
    #Regplot
        sns.regplot(data=tempDF, x='Sex-specific', y='Sex-pooled', color=tempD2[sex],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Sex-pooled '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF['Sex-specific'], tempDF['Sex-pooled'])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(sex, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Sex-specific '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

## 3. model predictors

### 3-1. clean beta-coeficiesnt dataframaea

In [ ]:
tempD1 = bcoefD.copy()

tempD2 = {}
for model in tempD1.keys():
    tempDF = tempD1[model]

    #Calculate standard statistic
    tempS1 = tempDF.mean(axis=1)
    tempS1.name = 'Mean'
    tempS2 = tempDF.std(axis=1, ddof=1)#Sample standard deviation
    tempS2.name = 'SD'
    tempS3 = (tempDF==0.0).sum(axis=1)
    tempS3.name = 'nZeros'
    tempDF = pd.concat([tempDF, tempS1, tempS2, tempS3], axis=1)

    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_ChemAge-LASSO_'
    fileName = 'bcoefs-'+model+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    tempD2[model] = tempDF
    
    #Check
    tempDF2 = tempDF.loc[['Intercept']]#Retrieve as pd.DataFrame
    tempDF = tempDF.drop(index=['Intercept'])
    print(model+' model:')
    print(' - Variables:', len(tempDF))
    tempDF1 = tempDF.loc[tempDF['nZeros']!=10]
    print(' - Variables with non-zero beta-coefficient:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF.loc[tempDF['nZeros']==0]
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF1.sort_values(by='Mean', ascending=False)
    display(tempDF1)
    display(tempDF2)#Intercept
    print('')

bcoefD = tempD2#Update

### 3-2 valiables retained in all models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Variables with non-zero beta-coefficient in all models
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:')
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF))
    
    #Prepare DF for plot
    tempDF = tempDF.sort_values(by='Mean', ascending=False)
    tempL = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    
    #Plot the robustly retained variables
    sns.set(style='ticks', context='talk')
    japanize_matplotlib.japanize()
    plt.figure(figsize=(18, len(tempL)/3))
    sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                dodge=False, color=tempD2[model], saturation=1,
                showcaps=True, notch=False, showfliers=False)
    sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                  markers='x', color='black',
                  dodge=False, join=False, estimator=np.mean, ci=None)#Add only the mean
    p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL, dodge=False, jitter=0.3,
                      size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    #p.grid(axis='x', linestyle='--', color='black')
    p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
    sns.despine()
    p.set(xlim=(-3.5, 7), xticks=np.arange(-3, 7.1, 1))#Fixed across models
    p.set(ylim=(-0.5, len(tempL)-0.5), xticks=np.arange(-3, 7.1, 1))
    ##Change default dull line color of sns.boxplot (saturation parameter is for patch)
    for line in p.get_lines():
        line.set_color('black')
    for box in p.artists:
        box.set_edgecolor('black')
    ##Add background color
    for row_i in range(len(tempL)):
        if row_i%2==0:
            plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
    p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
          title=tempD3[model]+' model (robust predictors)')
    plt.show()
    print('')

### 3-3. Comparison between sex-pooled and sex-specific models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare the robust predictors
tempD = {}
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempD[model] = set(tempDF.index.tolist())

#Venn diagram
sns.set(style='ticks',  context='talk')
fig, ax = plt.subplots(figsize=(4, 4))
venn(tempD, fmt='{size:,}', cmap=list(tempD2.values()), legend_loc=None, ax=ax)
plt.setp(ax, ylim=(0.025, 0.925))#Otherwise, weird space...
##Add legend annotation
x_coord = [0.3, 0.7, 0.65]
y_coord = [0.75, 0.75, 0.25]
h_align = ['right', 'left', 'left']
v_align = ['bottom', 'bottom', 'top']
for i in range(len(tempD3)):
    key = list(tempD3.keys())[i]
    total = f'{len(tempD[key]):,}'
    ax.text(x_coord[i], y_coord[i], tempD3[key]+'\n('+total+' items)',
            fontsize='small', multialignment='center',
            horizontalalignment=h_align[i], verticalalignment=v_align[i],
            bbox={'boxstyle':'round', 'facecolor':tempD2[key], 'pad':0.2, 'alpha':0.5})
title = 'Robustly retained items'
ax.set_title(title, fontsize='medium')
plt.show()

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare union of the robust predictors
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempL.append(tempDF.index.to_series())
tempI = pd.concat(tempL, axis=0)
tempL1 = tempI.unique().tolist()
print('-> Union:', len(tempL1))

#Prepare DF for plot
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Take the target variables
    tempDF = tempDF.loc[tempL1]
    
    #Prepare DF for plot
    if model=='SexPooled':
        tempDF = tempDF.sort_values(by='Mean', ascending=False)
        tempL2 = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    tempDF['Model'] = model
    tempL.append(tempDF)
tempDF = pd.concat(tempL, axis=0)

#Plot the target variables
sns.set(style='ticks', context='talk')
japanize_matplotlib.japanize()
plt.figure(figsize=(10, len(tempL2)/3))
p = sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
            hue='Model', hue_order=tempD2.keys(), dodge=True, palette=tempD2, saturation=1,
            showcaps=True, notch=False, showfliers=False)
#sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#              hue='Model', hue_order=tempD2.keys(), dodge=True, markers='x', color='black',
#              join=False, estimator=np.mean, ci=None)#Add only the mean
#p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#                  hue='Model', hue_order=tempD2.keys(), dodge=True, jitter=0.3,
#                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
#p.grid(axis='x', linestyle='--', color='black')
p.legend(bbox_to_anchor=(1.02, 1),loc='upper left')
p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
sns.despine()
p.set(xlim=(-7.5, 5), xticks=np.arange(-7, 5.1, 1))#Fixed across models
p.set(ylim=(-0.5, len(tempL2)-0.5), xticks=np.arange(-7, 5.1, 1))
##Change default dull line color of sns.boxplot (saturation parameter is for patch)
for line in p.get_lines():
    line.set_color('black')
for box in p.artists:
    box.set_edgecolor('black')
##Add background color
for row_i in range(len(tempL2)):
    if row_i%2==0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
      title=tempD3[model]+' model (robust predictors)')
plt.show()

### 3-4. Explained variance in CAge by the variable retained in ≥1 model

OLS linear regression model for significance: Age \~ b0 + b1\*Variable + b2\*Sex + b3\*BMI  
Univariate model: Age ~ b0 + b1\*Variable  

Of note, biological age is included for comparison. As well as LASSO models, unstandardized dependent variable is used to set the same beta-coefficient scale.  

In [ ]:

tempD1 = bcoefD
tempD2 = metD
tempD3 = predictD
tempD4 = cohortD
covarL = ['Sex', 'BMI']#Covariates (for now)
formula1 = yvar+' ~ Variable'#Univariate model
formula2 = yvar+' ~ Variable + C(Sex) + BMI'#Full model
formula3 = yvar+' ~ Variable + BMI'#Full model for the case that sex is invariant
tempD5 = {'SexPooled':formula2, 'Female':formula3, 'Male':formula3}
topX = 30

for metric, yvar_model in met.items():
    print('============================='+yvar_model+'=============================')
    for model in tempD1.keys():
        print(model+' model:')
    
        #Variables with non-zero beta-coefficient at least one of the models
        tempDF = tempD1[model]
        tempL = tempDF.loc[tempDF['nZeros']==10].index.tolist()#To be excluded
    
    #Prepare DF for regressions
        tempDF1 = tempD2[model].drop(columns=tempL)
    ##Add model predictions for comparison
        tempDF = tempD3[model]
        tempDF1[yvar_model+' model'] = tempDF[yvar_model]
        tempL = tempDF1.columns.tolist()#Save independent variables (including model) for regression-loop
    ##Add covariates
        tempDF = tempD4[model]
        tempDF = tempDF[covarL]
        tempDF1 = pd.merge(tempDF1, tempDF, left_index=True, right_index=True, how='left')
    ##Z-score transformation
        tempDF2 = tempDF1.select_dtypes(include=[np.number])
        scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
        tempA = scaler.fit_transform(tempDF2)#Column direction
        tempDF2 = pd.DataFrame(data=tempA, index=tempDF2.index, columns=tempDF2.columns)
    ###Recover categorical covariates
        tempDF3 = tempDF1.select_dtypes(exclude=[np.number])
        tempDF1 = pd.merge(tempDF2, tempDF3, left_index=True, right_index=True, how='left')
        ##Add unstandardized dependent variable
        tempDF = tempD4[model]
        tempS = tempDF[yvar]
        tempDF1 = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    #Perform OLS linear regression
        tempL1 = []#For sample size
        tempL2 = []#For R2 in the univariate model
        tempL3 = []#For R2
        tempL4 = []#For beta-coefficient
        tempL5 = []#For SE of beta-coefficient
        tempL6 = []#For t-statistic
        tempL7 = []#For residual degrees of freedom
        tempL8 = []#For P-value
        t_start = time.time()
        for var in tempL:
        #Rename independent variable
            tempDF = tempDF1.rename(columns={var: 'Variable'})
        #Add a constant for the intercept -> Similar to R, smf automatically add a constant
        
        #Save sample size
            tempL1.append(len(tempDF))
        
        #Fit univariate model
            fit_res = smf.ols(formula1, data=tempDF).fit()
        #Save R2 [%]
            tempL2.append(fit_res.rsquared*100)

        #Fit full model
            formula = tempD5[model]
            fit_res = smf.ols(formula, data=tempDF).fit()
        #Save R2 [%]
            tempL3.append(fit_res.rsquared*100)
        #Save beta-coefficient of the variable
            tempL4.append(fit_res.params['Variable'])
            tempL5.append(fit_res.bse['Variable'])
        #Save t-statistic of the variable
            tempL6.append(fit_res.tvalues['Variable'])
        #Save residual degrees of freedom
            tempL7.append(int(fit_res.df_resid))
        #Save P-value of the variable
            tempL8.append(fit_res.pvalues['Variable'])
        t_elapsed = time.time() - t_start
        print('Elapsed time for', len(tempL), 'OLS linear regressions (including '+yvar_model+'):',
              round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
        
    #Clean the results
        tempDF1 = pd.DataFrame({'N':tempL1, 'UnivarR2':tempL2, 'R2':tempL3,
                            'OLSbcoef':tempL4, 'OLSbcoefSE':tempL5,
                            'tStat':tempL6, 'DoF':tempL7,  'Pval':tempL8},
                           index=pd.Index(tempL, name='Variable'))
    ##P-value adjustment by using Benjamini–Hochberg method
        tempDF1['AdjPval'] = multi.multipletests(tempDF1['Pval'], alpha=0.05, method='fdr_bh',
                                             is_sorted=False, returnsorted=False)[1]
    ##Add the LASSO results
        tempDF = tempD1[model]
        tempDF1['LASSObcoef'] = tempDF.loc[tempDF.index.isin(tempL)]['Mean']#NaN in model reference
        tempDF1['LASSOnZeros'] = tempDF.loc[tempDF.index.isin(tempL)]['nZeros']#NaN in model reference
        tempDF1 = tempDF1.sort_values(by='AdjPval', ascending=True)
    
    #Save the cleaned DF
    #fileDir = './ExportData/67k/'
    #ipynbName = '260907_TMM-BAge_MetAge-LASSO-CommCohort67K-ver2k_'
    #fileName = 'OLS-'+model+'.tsv'
    #tempDF1.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    #Extact significant variables
        tempDF = tempDF1[tempDF1['AdjPval']<0.05]
        print('Variables significantly associated with '+yvar+' (FDR < 0.05):', len(tempDF)-1)#model reference
    ##Top X (+ model reference)
        tempDF = tempDF.iloc[0:(topX+1)]
        display(tempDF)
        ##Category and color
        tempL = []
        for row_n in tempDF.index:
            if row_n==yvar_model+' model':
                tempL.append('Positive association')
            else:
                bcoef = tempDF.loc[row_n, 'OLSbcoef']
                if bcoef>0:
                    tempL.append('Positive association')
                elif bcoef<0:
                    tempL.append('Negative association')
                else:
                    tempL.append('Error?')
        tempDF['Association'] = tempL
        tempD = {'Positive association':'tab:red', 'Negative association':'tab:blue'}
         ##Plot
        tempDF = tempDF.reset_index()
        sns.set(style='ticks',  context='talk')
        plt.figure(figsize=(10, 4))
        p = sns.barplot(data=tempDF, y='UnivarR2', x='Variable', hue='Association', hue_order=tempD.keys(),
                    dodge=False, palette=tempD, edgecolor='k')
        p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))#Fixed across models
        p.grid(axis='y', linestyle='--', color='black')
        sns.despine()
        plt.xticks(rotation=90, horizontalalignment='center')
        plt.xlabel('')
        plt.ylabel('Explained variance in '+yvar+' [%]',fontsize=15)
        plt.legend(loc='upper right')
        japanize_matplotlib.japanize()
        plt.show()
        print('')

# 4. LaboAge

## 1. data preparation

### 1-1. import cohort and platform

In [ ]:
#import RF met HC
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge_SpeAge_'
fileName = 'labotestDF-MF.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260930_TMM-BAge-CommCohort67K-ver1_'
fileName = 'demographicsDF.tsv'

tempDF=pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF

### 1-2. stratification with sex

In [ ]:
#Stratify the study cohort with sex
cohortDF_F = cohortDF.loc[cohortDF['Sex']=='Female']
cohortDF_M = cohortDF.loc[cohortDF['Sex']=='Male']
cohortDF_P = cohortDF#Not copy just rename
cohortD = {'SexPooled':cohortDF_P, 'Female':cohortDF_F, 'Male':cohortDF_M}

for cohort in cohortD.keys():
    tempDF = cohortD[cohort]
    print(cohort+': ', len(tempDF))

In [ ]:
tempDF1 = metDF
tempD1 = cohortD

#Prepare data per sex-stratified cohort
tempD2 = {}
for cohort in tempD1.keys():
    tempI = tempD1[cohort].index
    tempDF = tempDF1.loc[tempDF1.index.isin(tempI)]
    print(cohort+': ', len(tempDF))
    tempD2[cohort] = tempDF

metD = tempD2

### 1-3 split cohort

In [ ]:
tempD1 = cohortD
nmodels = 10#Fix throughout this notebook
random.seed(123)#For reproducibility (while checking the following independency)

tempD2 = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]
    
    #Randomize the row order
    tempL = tempDF1.index.tolist()
    tempL = random.sample(tempL, len(tempL))
    tempDF = tempDF1.loc[tempL]
    
    #Split cohort to define the training and testing (hold-out) sets
    tempL1 = np.array_split(tempDF, nmodels)#List of DFs
    ##Save only the testing set assignment
    tempL2 = []
    for model_k in range(nmodels):
        tempDF = tempL1[model_k]
        model_n = 'Model'+str(model_k+1).zfill(2)
        tempS = pd.Series(np.repeat(model_n, len(tempDF)), index=tempDF.index, name='Testing')
        tempL2.append(tempS)
    tempS = pd.concat(tempL2, axis=0)
    
    #Add to the original metadata
    tempDF = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    tempD2[cohort] = tempDF
    print(cohort)
    display(tempDF)
    display(tempDF['Testing'].value_counts())
    print('')

cohortD = tempD2#Update

### 1-4 check independency about sets

In [ ]:
tempD1 = cohortD
tempL0 = ['Testing', 'Sex', 'Age', 'BMI']#Set only the variables used in the study

for cohort in tempD1.keys():
    tempDF = tempD1[cohort]
    tempDF = tempDF[tempL0]
    
    tempL1 = []#For variable name
    tempL2 = []#Test name
    tempL3 = []#For degrees of freedom
    tempL4 = []#For test statistic
    tempL5 = []#For P-value
    
    #Categorical variables
    tempL = tempDF.select_dtypes(exclude=[np.number]).columns.tolist()
    for col_n in tempL:
        if col_n!='Testing':
            if (col_n=='Sex')&(cohort in ['Female', 'Male']):
                continue
            else:
                tempDF1 = pd.crosstab(tempDF[col_n], tempDF['Testing'])
                #Pearson's chi-squared test
                chi2, pval, dof, tempA = stats.chi2_contingency(tempDF1, correction=False)
                tempL1.append(col_n)
                tempL2.append('Pearson\'s chi-squared test')
                tempL3.append(dof)
                tempL4.append(chi2)
                tempL5.append(pval)
    
    #Numeric variables
    tempL = tempDF.select_dtypes(include=[np.number]).columns.tolist()
    for col_n in tempL:
        #ANOVA
        formula = col_n+' ~ C(Testing)'
        model = smf.ols(formula, data=tempDF).fit()
        tempDF1 = anova_lm(model, typ=1)#ANOVA type doesn't matter this case
        tempL1.append(col_n)
        tempL2.append('ANOVA')
        dof1 = tempDF1['df'].astype('int64').loc['C(Testing)']#Between-groups
        dof2 = tempDF1['df'].astype('int64').loc['Residual']#Within-groups
        tempL3.append((dof1, dof2))
        tempL4.append(tempDF1['F'].loc['C(Testing)'])
        tempL5.append(tempDF1['PR(>F)'].loc['C(Testing)'])

    #Summarize results as a single DF
    tempDF = pd.DataFrame({'Variable':tempL1,
                           'StatisticalTest':tempL2,
                           'N':len(tempDF),
                           'DoF':tempL3,
                           'Statistic':tempL4,
                           'Pval':tempL5})
    tempDF = tempDF.set_index('Variable')
    #P-value adjustment by using Benjamini–Hochberg method
    tempDF['AdjPval'] = multi.multipletests(tempDF['Pval'], alpha=0.05, method='fdr_bh',
                                            is_sorted=False, returnsorted=False)[1]
    print(cohort)
    display(tempDF)
    
    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261005_TMM-BAge_SpeAge-LASSO_'
    fileName = 'LAB_split-sets-independency-'+cohort+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

## 2. generate Bage model

### 2-1 lasso modeling with CV

In [ ]:
tempD1 = cohortD
tempD2 = metD
yvar = 'Age'#Fix throughout this notebook
ralphas = 0.05#Length of the regularization path (= alpha_min / alpha_max)
nalphas = 200#Number of alphas along the regularization path
ncvs = 10#Fix throughout this notebook
ncpus = -1#All processors

#Modeling per sex-stratified cohort
summaryD = {}#Initialize
bcoefD = {}#Initialize
predictD = {}#Initialize
scaler_meanD = {} #Inirialize
scaler_sdD={}
agegapD = {}
for cohort in tempD1.keys():
    tempDF1 = tempD1[cohort]#Dependent variable and info about testing set
    tempDF2 = tempD2[cohort]#Independent variables
    t_start1 = time.time()
    print(cohort+' start:', time.ctime(t_start1))
    
    #Modeling per model number (testing set)
    tempL1 = []#For alpha
    #PAge
    tempL2 = []#For the coefficient of determination R2
    tempL3 = []#For beta-coefficients
    tempL4 = []#For predictions
    tempL5=[] #For scaler means
    tempL6=[]#For scaler scales
    #other
    tempL7 = []#Agegap_reg
    tempL8 = []#BAge
    tempL9 = []#BAge R2score
    tempL10 = []#regression
    
    for model_k in range(nmodels):
        t_start2 = time.time()
        #Prepare DFs for model k
        model_n = 'Model'+str(model_k+1).zfill(2)
        ##Training set
        tempI = tempDF1.loc[tempDF1['Testing']!=model_n].index
        yDF_train = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_train = tempDF2.loc[tempI]
        ##Testing (hold-out) set
        tempI = tempDF1.loc[tempDF1['Testing']==model_n].index
        yDF_test = tempDF1.loc[tempI, [yvar]]#Note: dummy list to take as not pd.Series but DF
        xDF_test = tempDF2.loc[tempI]
        
        #Standardization (only for independent variables)
        ##Fit with the training set
        scaler_x = StandardScaler(copy=True, with_mean=True, with_std=True).fit(xDF_train)#Column direction
        
        #Save scaler parameters
        tempL5.append(pd.Series(scaler_x.mean_, index=xDF_train.columns,name=model_n))
        tempL6.append(pd.Series(scaler_x.scale_,index=xDF_train.columns,name=model_n))
        
        ##Z-score transformation using the training set-fitted scaler
        tempA = scaler_x.transform(xDF_train)
        xDF_train = pd.DataFrame(data=tempA, index=xDF_train.index, columns=xDF_train.columns)#Update
        tempA = scaler_x.transform(xDF_test)
        xDF_test = pd.DataFrame(data=tempA, index=xDF_test.index, columns=xDF_test.columns)#Update
        
        #Fit model with cross-validation using training dataset (i.e., internal training/validation datasets)
        model = LassoCV(eps=ralphas, n_alphas=nalphas, alphas=None, fit_intercept=True,
                        precompute='auto', cv=ncvs, n_jobs=ncpus)
        model.fit(xDF_train, yDF_train)
        
        #Save alpha
        tempL1.append(model.alpha_)#Penalization amount decided by cross-validation
        
        #Evaluation with the testing (hold-out) set
        tempL2.append(model.score(xDF_test, yDF_test))
        
        #Save beta-coefficients
        tempS = pd.Series(model.coef_,#w in the cost function formula
                          index=tempDF2.columns, name=model_n)
        tempS.loc['Intercept'] = model.intercept_#Independent term in decision function
        tempS.index.name = 'Variable'
        tempL3.append(tempS)

        #PAge prediction, agegap, BAge
        #train PAge prediction
        train_P = pd.Series(model.predict(xDF_train),index=xDF_train.index,name='PAge')
        #regression PAge~age -> calculate agegap
        train_g = pd.DataFrame({'Age': yDF_train['Age'],
                                'PAge':train_P})
        reg_fit = smf.ols('PAge ~ Age', data=train_g).fit()
        tempL10.append(pd.Series({'Intercept':reg_fit.params['Intercept'],
                                'Age': reg_fit.params['Age']}, name=model_n))

        #test PAge prediction
        test_P = pd.Series(model.predict(xDF_test),index=xDF_test.index,name='PAge')
        #test calculate agegap
        test_g = pd.DataFrame({'Age': yDF_test['Age'],
                             'PAge': test_P})
        test_g['AgeGap'] = (test_g['PAge'] - reg_fit.predict(test_g))
        #test: BAge = Age + AgeGap
        test_g['BAge'] = (test_g['Age'] + test_g['AgeGap'])
        
        #save PAge prediction
        tempL4.append(test_P)
        #save AgeGap and BAge
        tempL7.append(test_g['AgeGap'].rename('AgeGap'))
        tempL8.append(test_g['BAge'].rename('BAge'))

        #calculate BAge R2
        r2_b = 1- ((test_g['Age'] - test_g['BAge'])**2).sum()/(
            (test_g['Age'] - test_g['Age'].mean())**2).sum()
        tempL9.append(r2_b)
        
        t_elapsed2 = time.time() - t_start2
        print('- '+model_n+':',
              round(t_elapsed2//60), 'min', round(t_elapsed2%60, 1), 'sec')
    t_elapsed1 = time.time() - t_start1
    print('- Total elapsed time for '+str(nmodels)+' models with '+str(ncvs)+'-fold CV:',
          round(t_elapsed1//60), 'min', round(t_elapsed1%60, 1), 'sec')
    
    #Summarize results across models into a single DF
    tempL = ['Model'+str(model_k+1).zfill(2) for model_k in range(nmodels)]
    summaryDF = pd.DataFrame({'L1alpha':tempL1, 'R2score_P':tempL2, 'R2score_B':tempL9},
                             index=pd.Index(tempL, name='ModelID'))
    display(summaryDF)
    bcoefDF = pd.concat(tempL3, axis=1)
    display(bcoefDF)
    tempS1 = tempDF1[yvar]
    tempS2 = pd.concat(tempL4, axis=0)#PAge
    tempS3 = pd.concat(tempL7, axis=0)#agegap
    tempS4 = pd.concat(tempL8, axis=0)#BAge
    predictDF = pd.concat([tempS1, tempS2, tempS3, tempS4], axis=1)
    display(predictDF)
    SmeanDF= pd.concat(tempL5, axis=1)
    SmeanDF.columns.name = 'ModelID'
    display(SmeanDF)
    SsdDF = pd.concat(tempL6, axis=1)
    SsdDF.columns.name = 'ModelID'
    display(SsdDF)
    agegapDF = pd.DataFrame(tempL10)
    agegapDF.index.name = 'ModelID'
    agegapDF = agegapDF.T
    agegapDF.columns.name = 'ModelID'
    display(agegapDF)
    print('')
    
    #Save (Of note, bcoefDF is to be saved later)
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_LabAge-LASSO_'
    fileName = 'model-summary-'+cohort+'.tsv'
    summaryDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'bcoefs-'+cohort+'.tsv'
    #bcoefDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    fileName = 'predictions-'+cohort+'.tsv'
    predictDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_mean-'+cohort+'.tsv'
    #SmeanDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'scaler_sd-'+cohort+'.tsv'
    #SsdDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    #fileName = 'gap_regression-'+cohort+'.tsv'
    #agegapDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    
    summaryD[cohort] = summaryDF
    bcoefD[cohort] = bcoefDF
    predictD[cohort] = predictDF
    scaler_meanD[cohort]= SmeanDF
    scaler_sdD[cohort] = SsdDF
    agegapD[cohort] = agegapDF

### 2-2. prediction accuracy

In [ ]:
#Summary
tempD1 = summaryD
tempD2 = predictD

met = {'R2score_P':'PAge', 'R2score_B':'BAge'}

for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        print(model+' model')
        tempDF1 = tempD1[model]
        tempS = tempDF1[metric]
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SD]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1))#Sample standard deviation
        print(' - Out-of-sample R2_'+yvar_model+' [Mean ± SEM]:',
              np.mean(tempS), '±', np.std(tempS, ddof=1)/np.sqrt(len(tempS)))
        tempDF2 = tempD2[model]
        print(' - Observed vs. predicted '+yvar+': (Pearson\'s r, P) =',
              stats.pearsonr(tempDF2[yvar], tempDF2[yvar_model]))
        tempDF1 = tempDF1.describe()
        tempDF2 = tempDF2.describe()
        tempDF = pd.merge(tempDF1, tempDF2, left_index=True, right_index=True, how='left')
        display(tempDF)

In [ ]:
tempD1 = summaryD
met = {'R2score_P':'PAge', 'R2score_B':'BAge'}
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}

#Prepare DF
tempD = {}
for metric, yvar_model in met.items():
    print('=========================='+yvar_model+'=================================')

    for model in tempD1.keys():
        tempDF = tempD1[model]
        tempD[model] = tempDF[metric].tolist()
    tempDF = pd.DataFrame(tempD)
    display(tempDF.describe())
    tempDF = tempDF.melt(var_name='Model', value_name='R2', value_vars=tempDF.columns)

    #Plot
    sns.set(style='ticks',  context='talk')
    plt.figure(figsize=(2, 3))
    sns.barplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
            hue='Model', hue_order=tempD2.keys(), dodge=False, palette=tempD2,
            ci=95, capsize=0.4, errwidth=1.5, errcolor='black', edgecolor='black')
    p = sns.stripplot(data=tempDF, y='R2', x='Model', order=tempD2.keys(),
                  hue='Model', hue_order=tempD2.keys(), dodge=False, jitter=0.3,
                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    sns.despine()
    plt.setp(p.get_xticklabels(), rotation=70,
         horizontalalignment='right', verticalalignment='center', rotation_mode='anchor')
    plt.ylabel('Out-of-sample '+r'$R^2$_'+yvar_model, fontsize=13)
    plt.xlabel('')
    plt.show()

In [ ]:
tempD1 = predictD

for metric, yvar_model in met.items():
#Prepare plot range
    range_min = np.min([df[var].min() for df in tempD1.values() for var in [yvar, yvar_model]])
    range_max = np.max([df[var].max() for df in tempD1.values() for var in [yvar, yvar_model]])
    offset_rate = 0.05
    offset = (range_max-range_min)*offset_rate
    range_min = range_min - offset#Update
    range_max = range_max + offset#Update
    xymin = 20
    xymax = 100
    xyinter = 20

#Plot measured vs. predicted per model
    tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = 'Age [years]'
    sns.set(style='ticks', context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(10, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        model = list(tempD2.keys())[ax_i]
        tempDF = tempD1[model]
    #Regplot
        sns.regplot(data=tempDF, y=yvar_model, x=yvar, color=tempD2[model],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Predicted '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF[yvar], tempDF[yvar_model])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(model, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Measured '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

#Plot difference b/w sex-specific and sex-pooled models
    tempD2 = {'Female':'tab:red', 'Male':'tab:blue'}
    axis_label = yvar_model+' [years]'
    sns.set(style='ticks',  context='talk')
    fig, axes = plt.subplots(nrows=1, ncols=len(tempD2), figsize=(6.5, 3), sharex=True, sharey=True,
                         gridspec_kw={'hspace':0.1, 'wspace':0.1})
##Set axis range first; otherwise, regression line can be truncated differently
    plt.setp(axes, xlim=(range_min, range_max), xticks=np.arange(xymin, xymax, xyinter))
    plt.setp(axes, ylim=(range_min, range_max), yticks=np.arange(xymin, xymax, xyinter))
    for ax_i, ax in enumerate(axes.flat):
        sex = list(tempD2.keys())[ax_i]
    #Prepare DF
        tempS1 = tempD1[sex][yvar_model]
        tempS1.name = 'Sex-specific'
        tempS2 = tempD1['SexPooled'][yvar_model]
        tempS2.name = 'Sex-pooled'
        tempDF = pd.merge(tempS1, tempS2, left_index=True, right_index=True, how='inner')
    #Regplot
        sns.regplot(data=tempDF, x='Sex-specific', y='Sex-pooled', color=tempD2[sex],
                scatter=True, fit_reg=True, ci=95, truncate=False, marker='o',
                scatter_kws={'alpha':0.2, 'edgecolor':'k', 's':25}, ax=ax)
    #Draw Y=X as reference
        ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='black', linestyle=(0, (1, 2)), zorder=0)
    #Axis title
        if ax_i%len(tempD2)==0:
            plt.setp(ax, xlabel='', ylabel='Sex-pooled '+axis_label)
        else:
            plt.setp(ax, xlabel='', ylabel='')
            plt.setp(ax.get_yticklabels(), visible=False)
    #Annotate Pearson's correlation statistics
        coef, pval = stats.pearsonr(tempDF['Sex-specific'], tempDF['Sex-pooled'])
        coef_text = str(Decimal(str(coef)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))
        pval_text = convert_pval(pval).replace('=', '= ').replace('<', '< ')#Add a white space
        text = 'Pearson\'s '+r'$r$'+' = '+coef_text+'\n'+r'$P$'+' '+pval_text
        ax.annotate(text, xy=(0.025, 0.975), xycoords='axes fraction',
                horizontalalignment='left', verticalalignment='top',
                multialignment='left', fontsize='x-small', color='k')
    #Facet label
        ax.set_title(sex, {'fontsize':'medium'})
    #Save position to generate facet and legend later
        if ax_i ==0:
            ax0_pos = ax.get_position().bounds
        elif ax_i==len(tempD2)-1:
            ax1_pos = ax.get_position().bounds
    sns.despine()
    fig.text(x=(ax0_pos[0]+(ax1_pos[0]+ax1_pos[2]))/2, y=ax0_pos[1]-ax0_pos[3]*0.2,#Minor manual adjustment
         s='Sex-specific '+axis_label, fontsize='medium',
         verticalalignment='top', horizontalalignment='center', rotation='horizontal')
    plt.show()

## 3. model predictors

### 3-1. clean beta-coeficiesnt dataframaea

In [ ]:
tempD1 = bcoefD.copy()

tempD2 = {}
for model in tempD1.keys():
    tempDF = tempD1[model]

    #Calculate standard statistic
    tempS1 = tempDF.mean(axis=1)
    tempS1.name = 'Mean'
    tempS2 = tempDF.std(axis=1, ddof=1)#Sample standard deviation
    tempS2.name = 'SD'
    tempS3 = (tempDF==0.0).sum(axis=1)
    tempS3.name = 'nZeros'
    tempDF = pd.concat([tempDF, tempS1, tempS2, tempS3], axis=1)

    #Save
    fileDir = './ExportData/Savemodel/specific/LASSO/'
    ipynbName = '261019_TMM-BAge_LabAge-LASSO_'
    fileName = 'bcoefs-'+model+'.tsv'
    tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    tempD2[model] = tempDF
    
    #Check
    tempDF2 = tempDF.loc[['Intercept']]#Retrieve as pd.DataFrame
    tempDF = tempDF.drop(index=['Intercept'])
    print(model+' model:')
    print(' - Variables:', len(tempDF))
    tempDF1 = tempDF.loc[tempDF['nZeros']!=10]
    print(' - Variables with non-zero beta-coefficient:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF.loc[tempDF['nZeros']==0]
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF1),
          '(', len(tempDF1)/len(tempDF)*100, '%)')
    tempDF1 = tempDF1.sort_values(by='Mean', ascending=False)
    display(tempDF1)
    display(tempDF2)#Intercept
    print('')

bcoefD = tempD2#Update

### 3-2 valiables retained in all models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Variables with non-zero beta-coefficient in all models
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:')
    print(' - Variables with non-zero beta-coefficient in all models:', len(tempDF))
    
    #Prepare DF for plot
    tempDF = tempDF.sort_values(by='Mean', ascending=False)
    tempL = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    
    #Plot the robustly retained variables
    sns.set(style='ticks', context='talk')
    japanize_matplotlib.japanize()
    plt.figure(figsize=(18, len(tempL)/3))
    sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                dodge=False, color=tempD2[model], saturation=1,
                showcaps=True, notch=False, showfliers=False)
    sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL,
                  markers='x', color='black',
                  dodge=False, join=False, estimator=np.mean, ci=None)#Add only the mean
    p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL, dodge=False, jitter=0.3,
                      size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
    #p.grid(axis='x', linestyle='--', color='black')
    p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
    sns.despine()
    p.set(xlim=(-4.5, 6), xticks=np.arange(-4, 6.1, 1))#Fixed across models
    p.set(ylim=(-0.5, len(tempL)-0.5), xticks=np.arange(-4, 6.1, 1))
    ##Change default dull line color of sns.boxplot (saturation parameter is for patch)
    for line in p.get_lines():
        line.set_color('black')
    for box in p.artists:
        box.set_edgecolor('black')
    ##Add background color
    for row_i in range(len(tempL)):
        if row_i%2==0:
            plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
    p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
          title=tempD3[model]+' model (robust predictors)')
    plt.show()
    print('')

### 3-3. Comparison between sex-pooled and sex-specific models

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare the robust predictors
tempD = {}
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempD[model] = set(tempDF.index.tolist())

#Venn diagram
sns.set(style='ticks',  context='talk')
fig, ax = plt.subplots(figsize=(4, 4))
venn(tempD, fmt='{size:,}', cmap=list(tempD2.values()), legend_loc=None, ax=ax)
plt.setp(ax, ylim=(0.025, 0.925))#Otherwise, weird space...
##Add legend annotation
x_coord = [0.3, 0.7, 0.65]
y_coord = [0.75, 0.75, 0.25]
h_align = ['right', 'left', 'left']
v_align = ['bottom', 'bottom', 'top']
for i in range(len(tempD3)):
    key = list(tempD3.keys())[i]
    total = f'{len(tempD[key]):,}'
    ax.text(x_coord[i], y_coord[i], tempD3[key]+'\n('+total+' items)',
            fontsize='small', multialignment='center',
            horizontalalignment=h_align[i], verticalalignment=v_align[i],
            bbox={'boxstyle':'round', 'facecolor':tempD2[key], 'pad':0.2, 'alpha':0.5})
title = 'Robustly retained items'
ax.set_title(title, fontsize='medium')
plt.show()

In [ ]:
tempD1 = bcoefD
tempD2 = {'SexPooled':'tab:green', 'Female':'tab:red', 'Male':'tab:blue'}
tempD3 = {'SexPooled':'Sex-pooled', 'Female':'Female-specific', 'Male':'Male-specific'}

#Prepare union of the robust predictors
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    tempDF = tempDF.loc[tempDF['nZeros']==0]
    print(model+' model:', len(tempDF))
    tempL.append(tempDF.index.to_series())
tempI = pd.concat(tempL, axis=0)
tempL1 = tempI.unique().tolist()
print('-> Union:', len(tempL1))

#Prepare DF for plot
tempL = []
for model in tempD1.keys():
    tempDF = tempD1[model]
    tempDF = tempDF.drop(index=['Intercept'])
    
    #Take the target variables
    tempDF = tempDF.loc[tempL1]
    
    #Prepare DF for plot
    if model=='SexPooled':
        tempDF = tempDF.sort_values(by='Mean', ascending=False)
        tempL2 = tempDF.index.tolist()#Save order
    tempDF = tempDF.drop(columns=['Mean', 'SD', 'nZeros'])
    tempDF = tempDF.reset_index().melt(var_name='ModelID', value_name='bcoef', id_vars=['Variable'])
    tempDF['Model'] = model
    tempL.append(tempDF)
tempDF = pd.concat(tempL, axis=0)

#Plot the target variables
sns.set(style='ticks', context='talk')
japanize_matplotlib.japanize()
plt.figure(figsize=(15, len(tempL2)/3))
p = sns.boxplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
            hue='Model', hue_order=tempD2.keys(), dodge=True, palette=tempD2, saturation=1,
            showcaps=True, notch=False, showfliers=False)
#sns.pointplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#              hue='Model', hue_order=tempD2.keys(), dodge=True, markers='x', color='black',
#              join=False, estimator=np.mean, ci=None)#Add only the mean
#p = sns.stripplot(data=tempDF, y='Variable', x='bcoef', order=tempL2,
#                  hue='Model', hue_order=tempD2.keys(), dodge=True, jitter=0.3,
#                  size=5, edgecolor='black', color='gray', linewidth=1, alpha=0.4)
#p.grid(axis='x', linestyle='--', color='black')
p.legend(bbox_to_anchor=(1.02, 1),loc='upper left')
p.axvline(x=0, **{'linestyle':'--', 'color':'black', 'zorder':0.0})
sns.despine()
p.set(xlim=(-8.5, 4), xticks=np.arange(-8, 4.1, 1))#Fixed across models
p.set(ylim=(-0.5, len(tempL2)-0.5), xticks=np.arange(-8, 4.1, 1))
##Change default dull line color of sns.boxplot (saturation parameter is for patch)
for line in p.get_lines():
    line.set_color('black')
for box in p.artists:
    box.set_edgecolor('black')
##Add background color
for row_i in range(len(tempL2)):
    if row_i%2==0:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='gray', alpha=0.4, zorder=0)
p.set(ylabel='', xlabel=r'$\beta$'+'-coefficient in LASSO model\n[years per s.d.]',
      title=tempD3[model]+' model (robust predictors)')
plt.show()

### 3-4. Explained variance in CAge by the variable retained in ≥1 model

OLS linear regression model for significance: Age \~ b0 + b1\*Variable + b2\*Sex + b3\*BMI  
Univariate model: Age ~ b0 + b1\*Variable  

Of note, biological age is included for comparison. As well as LASSO models, unstandardized dependent variable is used to set the same beta-coefficient scale.  

In [ ]:

tempD1 = bcoefD
tempD2 = metD
tempD3 = predictD
tempD4 = cohortD
covarL = ['Sex', 'BMI']#Covariates (for now)
formula1 = yvar+' ~ Variable'#Univariate model
formula2 = yvar+' ~ Variable + C(Sex) + BMI'#Full model
formula3 = yvar+' ~ Variable + BMI'#Full model for the case that sex is invariant
tempD5 = {'SexPooled':formula2, 'Female':formula3, 'Male':formula3}
topX = 30

for metric, yvar_model in met.items():
    print('============================='+yvar_model+'=============================')
    for model in tempD1.keys():
        print(model+' model:')
    
        #Variables with non-zero beta-coefficient at least one of the models
        tempDF = tempD1[model]
        tempL = tempDF.loc[tempDF['nZeros']==10].index.tolist()#To be excluded
    
    #Prepare DF for regressions
        tempDF1 = tempD2[model].drop(columns=tempL)
    ##Add model predictions for comparison
        tempDF = tempD3[model]
        tempDF1[yvar_model+' model'] = tempDF[yvar_model]
        tempL = tempDF1.columns.tolist()#Save independent variables (including model) for regression-loop
    ##Add covariates
        tempDF = tempD4[model]
        tempDF = tempDF[covarL]
        tempDF1 = pd.merge(tempDF1, tempDF, left_index=True, right_index=True, how='left')
    ##Z-score transformation
        tempDF2 = tempDF1.select_dtypes(include=[np.number])
        scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
        tempA = scaler.fit_transform(tempDF2)#Column direction
        tempDF2 = pd.DataFrame(data=tempA, index=tempDF2.index, columns=tempDF2.columns)
    ###Recover categorical covariates
        tempDF3 = tempDF1.select_dtypes(exclude=[np.number])
        tempDF1 = pd.merge(tempDF2, tempDF3, left_index=True, right_index=True, how='left')
        ##Add unstandardized dependent variable
        tempDF = tempD4[model]
        tempS = tempDF[yvar]
        tempDF1 = pd.merge(tempDF1, tempS, left_index=True, right_index=True, how='left')
    
    #Perform OLS linear regression
        tempL1 = []#For sample size
        tempL2 = []#For R2 in the univariate model
        tempL3 = []#For R2
        tempL4 = []#For beta-coefficient
        tempL5 = []#For SE of beta-coefficient
        tempL6 = []#For t-statistic
        tempL7 = []#For residual degrees of freedom
        tempL8 = []#For P-value
        t_start = time.time()
        for var in tempL:
        #Rename independent variable
            tempDF = tempDF1.rename(columns={var: 'Variable'})
        #Add a constant for the intercept -> Similar to R, smf automatically add a constant
        
        #Save sample size
            tempL1.append(len(tempDF))
        
        #Fit univariate model
            fit_res = smf.ols(formula1, data=tempDF).fit()
        #Save R2 [%]
            tempL2.append(fit_res.rsquared*100)

        #Fit full model
            formula = tempD5[model]
            fit_res = smf.ols(formula, data=tempDF).fit()
        #Save R2 [%]
            tempL3.append(fit_res.rsquared*100)
        #Save beta-coefficient of the variable
            tempL4.append(fit_res.params['Variable'])
            tempL5.append(fit_res.bse['Variable'])
        #Save t-statistic of the variable
            tempL6.append(fit_res.tvalues['Variable'])
        #Save residual degrees of freedom
            tempL7.append(int(fit_res.df_resid))
        #Save P-value of the variable
            tempL8.append(fit_res.pvalues['Variable'])
        t_elapsed = time.time() - t_start
        print('Elapsed time for', len(tempL), 'OLS linear regressions (including '+yvar_model+'):',
              round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
        
    #Clean the results
        tempDF1 = pd.DataFrame({'N':tempL1, 'UnivarR2':tempL2, 'R2':tempL3,
                            'OLSbcoef':tempL4, 'OLSbcoefSE':tempL5,
                            'tStat':tempL6, 'DoF':tempL7,  'Pval':tempL8},
                           index=pd.Index(tempL, name='Variable'))
    ##P-value adjustment by using Benjamini–Hochberg method
        tempDF1['AdjPval'] = multi.multipletests(tempDF1['Pval'], alpha=0.05, method='fdr_bh',
                                             is_sorted=False, returnsorted=False)[1]
    ##Add the LASSO results
        tempDF = tempD1[model]
        tempDF1['LASSObcoef'] = tempDF.loc[tempDF.index.isin(tempL)]['Mean']#NaN in model reference
        tempDF1['LASSOnZeros'] = tempDF.loc[tempDF.index.isin(tempL)]['nZeros']#NaN in model reference
        tempDF1 = tempDF1.sort_values(by='AdjPval', ascending=True)
    
    #Save the cleaned DF
    #fileDir = './ExportData/67k/'
    #ipynbName = '260907_TMM-BAge_MetAge-LASSO-CommCohort67K-ver2k_'
    #fileName = 'OLS-'+model+'.tsv'
    #tempDF1.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)
    
    #Extact significant variables
        tempDF = tempDF1[tempDF1['AdjPval']<0.05]
        print('Variables significantly associated with '+yvar+' (FDR < 0.05):', len(tempDF)-1)#model reference
    ##Top X (+ model reference)
        tempDF = tempDF.iloc[0:(topX+1)]
        display(tempDF)
        ##Category and color
        tempL = []
        for row_n in tempDF.index:
            if row_n==yvar_model+' model':
                tempL.append('Positive association')
            else:
                bcoef = tempDF.loc[row_n, 'OLSbcoef']
                if bcoef>0:
                    tempL.append('Positive association')
                elif bcoef<0:
                    tempL.append('Negative association')
                else:
                    tempL.append('Error?')
        tempDF['Association'] = tempL
        tempD = {'Positive association':'tab:red', 'Negative association':'tab:blue'}
         ##Plot
        tempDF = tempDF.reset_index()
        sns.set(style='ticks',  context='talk')
        plt.figure(figsize=(10, 4))
        p = sns.barplot(data=tempDF, y='UnivarR2', x='Variable', hue='Association', hue_order=tempD.keys(),
                    dodge=False, palette=tempD, edgecolor='k')
        p.set(ylim=(0, 80), yticks=np.arange(0, 80, 15))#Fixed across models
        p.grid(axis='y', linestyle='--', color='black')
        sns.despine()
        plt.xticks(rotation=90, horizontalalignment='center')
        plt.xlabel('')
        plt.ylabel('Explained variance in '+yvar+' [%]',fontsize=15)
        plt.legend(loc='upper right')
        japanize_matplotlib.japanize()
        plt.show()
        print('')

# end of notebook